# YouTube Media Suite & Playlist Toolkit Pro -- Desktop Processing Studio
**High-Performance Multimedia YouTube Downloader, Audio Extractor, Metadata Inspector, Direct CDN Link Exporter & Media Player**  
**Developed by: mohatamim**

---

### Core Operations:
1. **Playlist Downloader (Alt+1)**: Batch download full or indexed playlist ranges with video quality selection (4K, 1440p, 1080p, 720p, 480p, 360p, Best) and container options (.mp4, .mkv, .webm).
2. **Single Video Downloader (Alt+2)**: Fast on-demand single video stream download with resolution presets.
3. **Batch Audio Extractor (Alt+3)**: Extract pristine audio streams to MP3 (320k/192k), M4A (AAC), FLAC (Lossless), or WAV, with embedded ID3 metadata tags and thumbnail album art.
4. **Metadata Inspector & Video Picker (Alt+4)**: Inspect playlist items non-blockingly in an interactive table preview without downloading. Check and selectively download chosen items.
5. **Direct CDN Link Exporter (Alt+5)**: Extract raw direct GoogleVideo CDN streaming URLs with customizable stream quality dropdown (1080p, 720p, 480p, 360p, 4K, Audio only) for external download managers.

---

### Advanced Pro Features:
- **Individual Video Progress Monitor**: Click on the main progress bar (or the Task Monitor button) to open a real-time modal tracking each video with its own individual progress bar, status badge (QUEUED, DOWNLOADING, POSTPROCESSING, COMPLETED, ERROR), speed in MB/s, and downloaded bytes.
- **Custom Professional Media Player**: Integrated audio and video preview player featuring animated audio spectrum equalizer visualizer, interactive seek slider, volume control, -10s/+10s jump buttons, loop mode, and external player launching for both streaming URLs and local files.
- **Strict Byte-Based Calculations**: All file sizes, speed readouts, and estimated totals are calculated strictly in standard Bytes (B, KB, MB, GB, TB with base 1024), completely eliminating bit/byte confusion.
- **Simultaneous / Concurrent Downloads**: Choose how many videos to download together (1 Sequential, 2, 3, 4, 5, 8, or 10 parallel threads) with combined speed metrics and smooth progress tracking.
- **CDN Stream Quality Selector**: Choose specific stream resolution (1080p, 720p, 480p, 360p, 4K, Audio only) when extracting direct GoogleVideo links.
- **Smart Queue Deduplication**: 1-click duplicate video detector and remover scanning video IDs and URLs.
- **Real-Time Live Search Filter**: Search queue videos dynamically by title or uploader name.
- **Download Speed Throttling**: Limit bandwidth usage (Unlimited, 10 MB/s, 5 MB/s, 2 MB/s, 1 MB/s).
- **Enhanced Visual Selection**: Selected items highlighted with deep navy background (#132D4B) and bright text.
- **Refined Styling & Ergonomics**: Clean input padding, modern dropdown styling, and hand2 cursor across all interactive controls.
- **Persistent Bottom Log Console**: Live log console and telemetry docked to bottom for uninterrupted status visibility.
- **Zero Emojis**: Clean, professional technical interface.

---

### Quick Start:
1. Run **Cell 2** below to verify environment dependencies (`yt-dlp` and `FFmpeg`).
2. Run **Cell 3** to launch the interactive Desktop GUI Dashboard!


In [1]:
# ==============================================================================
# CELL 2: ENVIRONMENT AND DEPENDENCY CHECKS
# ==============================================================================
import sys
import shutil
import subprocess

print("Checking required Python packages and FFmpeg...")

required_packages = [
    "yt-dlp>=2024.0.0",
    "imageio-ffmpeg>=0.4.9"
]

missing_packages = []
for pkg in required_packages:
    pkg_name = pkg.split(">=")[0].replace("-", "_")
    try:
        __import__(pkg_name)
    except ImportError:
        missing_packages.append(pkg)

if missing_packages:
    print(f"Installing missing packages: {', '.join(missing_packages)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print("[OK] All required packages installed.")
else:
    print("[OK] All required Python packages are installed.")

# Check FFmpeg binary
try:
    import imageio_ffmpeg
    ff_exe = imageio_ffmpeg.get_ffmpeg_exe()
    proc = subprocess.run([ff_exe, "-version"], capture_output=True, text=True)
    first_l = proc.stdout.splitlines()[0] if proc.stdout else "Detected"
    print(f"[OK] FFmpeg binary verified: {first_l}")
except Exception as e:
    print(f"[NOTE] FFmpeg check: {e}")

Checking required Python packages and FFmpeg...
[OK] All required Python packages are installed.
[OK] FFmpeg binary verified: ffmpeg version 7.1-essentials_build-www.gyan.dev Copyright (c) 2000-2024 the FFmpeg developers


In [2]:
# ==============================================================================
# CELL 3: YOUTUBE MEDIA SUITE & PLAYLIST TOOLKIT PRO DESKTOP GUI
# ==============================================================================
"""
==============================================================================
YouTube Media Suite & Playlist Toolkit Pro -- Desktop Processing Studio
Developed by: mohatamim
==============================================================================
Unified enterprise YouTube downloading, audio extraction, metadata inspection,
and direct streaming link generation suite combining:
  1. Playlist Downloader (Preview with Thumbnails, Individual Quality Settings, Batch Download)
  2. Single Video Downloader (On-demand fast stream downloader with resolution presets)
  3. Batch Audio Extractor (High-fidelity MP3 320k/192k, M4A, FLAC, WAV with metadata)
  4. Metadata Inspector & Video Picker (Flat non-blocking playlist inspection & selective batch download)
  5. Direct CDN Link Exporter (In-dashboard stream URL viewer & multi-link clipboard copy for IDM/aria2)

New & Upgraded Features:
  - Quality Selector for CDN Extractor: Choose exact resolution (1080p, 720p, 480p, 360p, 4K, Audio) for CDN stream extraction.
  - Highlighted Row Backgrounds: Checked items dynamically highlight with deep navy blue (#132D4B) vs dark slate (#0F172A).
  - Deduplication Engine: 1-click 'Remove Duplicates' scans video IDs and URLs, eliminating duplicate entries.
  - Real-Time Queue Filter: Instant keyword search box to filter videos by title or channel.
  - Aggregate Summary Badge: Real-time calculation of total duration and estimated download size for all selected items.
  - Download Speed Throttling: Optional bandwidth cap (Unlimited, 10 MB/s, 5 MB/s, 2 MB/s, 1 MB/s).
  - Simultaneous Downloads: User can choose how many videos to download together (1 Sequential, 2, 3, 4, 5, 8, 10 simultaneously) with thread-safe parallel ThreadPool execution and aggregate telemetry.
  - Open in Browser: Inspect and open any video directly in web browser.
  - Professional UI Design: Polished padding, typography, consistent colors, hand pointer cursors, zero emojis.
==============================================================================
"""

import os
import sys
import time
import shutil
import logging
import threading
import queue
import concurrent.futures
import json
import re
import io
import urllib.request
import webbrowser
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any, Set, Callable

# Ensure local directories and parent paths are in sys.path
for _p in [
    Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd(),
    Path.cwd(),
    Path.cwd().parent,
    Path(r"d:\New folder\PY"),
    Path(r"d:\New folder\PY\scripts")
]:
    _p_str = str(_p.resolve()) if hasattr(_p, "resolve") and _p.exists() else str(_p)
    if _p_str not in sys.path:
        sys.path.insert(0, _p_str)

import tkinter as tk
from tkinter import ttk, filedialog, messagebox

# Pillow for thumbnail processing
try:
    from PIL import Image, ImageTk, ImageDraw
    PILLOW_AVAILABLE = True
except ImportError:
    PILLOW_AVAILABLE = False

# Check optional windnd for OS drag-and-drop
try:
    import windnd
    WINDND_AVAILABLE = True
except ImportError:
    WINDND_AVAILABLE = False

# yt-dlp import with safety check
try:
    import yt_dlp
    YTDLP_AVAILABLE = True
except ImportError:
    yt_dlp = None
    YTDLP_AVAILABLE = False

# imageio_ffmpeg import for bundled ffmpeg
try:
    import imageio_ffmpeg
    IMAGEIO_FFMPEG_AVAILABLE = True
except ImportError:
    imageio_ffmpeg = None
    IMAGEIO_FFMPEG_AVAILABLE = False


# ==============================================================================
# SYSTEM & FFMPEG BINARY RESOLVER
# ==============================================================================

def resolve_ffmpeg() -> Tuple[Optional[str], Optional[str]]:
    """Locates the FFmpeg and FFprobe binary executables."""
    ffmpeg_path = None
    ffprobe_path = None

    if IMAGEIO_FFMPEG_AVAILABLE and imageio_ffmpeg:
        try:
            exe = imageio_ffmpeg.get_ffmpeg_exe()
            if exe and os.path.isfile(exe):
                ffmpeg_path = exe
                probe_cand = os.path.join(os.path.dirname(exe), "ffprobe.exe")
                if os.path.isfile(probe_cand):
                    ffprobe_path = probe_cand
        except Exception:
            pass

    if not ffmpeg_path:
        venv_ffmpeg = Path(r"d:\New folder\PY\.venv\Scripts\ffmpeg.exe")
        if venv_ffmpeg.is_file():
            ffmpeg_path = str(venv_ffmpeg)
            venv_probe = Path(r"d:\New folder\PY\.venv\Scripts\ffprobe.exe")
            if venv_probe.is_file():
                ffprobe_path = str(venv_probe)

    if not ffmpeg_path:
        sys_ffmpeg = shutil.which("ffmpeg")
        if sys_ffmpeg:
            ffmpeg_path = sys_ffmpeg
            sys_probe = shutil.which("ffprobe")
            if sys_probe:
                ffprobe_path = sys_probe

    return ffmpeg_path, ffprobe_path


FFMPEG_EXECUTABLE, FFPROBE_EXECUTABLE = resolve_ffmpeg()
FFMPEG_DIRECTORY = os.path.dirname(FFMPEG_EXECUTABLE) if FFMPEG_EXECUTABLE else None


# ==============================================================================
# UTILITY FORMATTERS
# ==============================================================================

class YtDlpLogger:
    """Redirects yt-dlp messages into the thread-safe UI queue."""
    def __init__(self, ui_queue: queue.Queue):
        self.ui_queue = ui_queue

    def debug(self, msg: str):
        if msg.startswith("[debug] "):
            return
        self.ui_queue.put(("log", f"[INFO] {msg.strip()}"))

    def info(self, msg: str):
        self.ui_queue.put(("log", f"[INFO] {msg.strip()}"))

    def warning(self, msg: str):
        self.ui_queue.put(("log", f"[WARNING] {msg.strip()}"))

    def error(self, msg: str):
        self.ui_queue.put(("log", f"[ERROR] {msg.strip()}"))


def format_bytes(b: Optional[float]) -> str:
    """Converts raw byte count into standard human-readable format (Bytes, KB, MB, GB, TB)."""
    if not b or b <= 0:
        return "0.0 MB"
    try:
        val = float(b)
    except (ValueError, TypeError):
        return "0.0 MB"
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if val < 1024.0:
            return f"{val:.1f} {unit}" if unit in ('B', 'KB') else f"{val:.2f} {unit}"
        val /= 1024.0
    return f"{val:.2f} PB"


def format_seconds(s: Optional[float]) -> str:
    """Converts seconds into HH:MM:SS or MM:SS format."""
    if not s or s <= 0:
        return "00:00"
    s = int(s)
    hrs = s // 3600
    mins = (s % 3600) // 60
    secs = s % 60
    if hrs > 0:
        return f"{hrs:02d}:{mins:02d}:{secs:02d}"
    return f"{mins:02d}:{secs:02d}"


def estimate_size_bytes(duration_sec: Optional[float], quality_str: str) -> int:
    """
    Estimates approximate media file size in BYTES (not bits) based on realistic YouTube bitrates.
    Bitrate (bits/sec) / 8 = Bytes per second.
    """
    if not duration_sec or duration_sec <= 0:
        return 0
    q = str(quality_str).lower().replace("p", "").strip()

    # Megabytes per minute (in actual Bytes):
    # 4K ~25 Mbps = 3.125 MB/s = ~187 MB/min
    # 1440p ~12 Mbps = 1.5 MB/s = ~90 MB/min
    # 1080p ~4.8 Mbps = 600 KB/s = ~36 MB/min
    # 720p ~2.4 Mbps = 300 KB/s = ~18 MB/min
    # 480p ~1.2 Mbps = 150 KB/s = ~9 MB/min
    # 360p ~650 kbps = 81 KB/s = ~4.8 MB/min
    # mp3 320k = 40 KB/s = ~2.4 MB/min
    # m4a 192k = 24 KB/s = ~1.44 MB/min
    # flac ~850 kbps = 106 KB/s = ~6.4 MB/min
    # wav 1411 kbps = 176.4 KB/s = ~10.58 MB/min
    mb_per_minute = {
        '2160': 185.0,
        '1440': 90.0,
        '1080': 36.0,
        '720': 18.0,
        '480': 9.0,
        '360': 4.8,
        'mp3': 2.4,
        'm4a': 1.44,
        'flac': 6.4,
        'wav': 10.58
    }
    rate_mb = mb_per_minute.get(q, 25.0)
    total_bytes = int((duration_sec / 60.0) * rate_mb * 1024 * 1024)
    return max(0, total_bytes)


def estimate_size(duration_sec: Optional[float], quality_str: str) -> str:
    """Estimates approximate media file size formatted in Bytes (KB, MB, GB)."""
    raw_bytes = estimate_size_bytes(duration_sec, quality_str)
    if raw_bytes <= 0:
        return "Unknown"
    return format_bytes(raw_bytes)


def parse_target_urls(raw_input: str) -> List[str]:
    """Parses newline-, comma-, or whitespace-delimited URLs into clean YouTube URLs."""
    urls = []
    for line in re.split(r'[\r\n,;]+', raw_input):
        line = line.strip()
        if not line:
            continue
        # Raw YouTube playlist ID (e.g. PL..., UU..., LL..., FL...)
        if re.match(r'^(PL|UU|LL|FL|RD)[a-zA-Z0-9_-]+$', line):
            urls.append(f"https://www.youtube.com/playlist?list={line}")
            continue
        # Raw 11-char video ID
        if re.match(r'^[a-zA-Z0-9_-]{11}$', line):
            urls.append(f"https://www.youtube.com/watch?v={line}")
            continue
        if line.startswith(('http://', 'https://')):
            urls.append(line)
        elif line.startswith(('www.', 'youtube.com', 'youtu.be')):
            urls.append('https://' + line)
        elif 'youtube.com' in line or 'youtu.be' in line:
            urls.append('https://' + line.lstrip('/'))
        else:
            urls.append(line)
    return urls


# ==============================================================================
# CORE DOWNLOADER ENGINE
# ==============================================================================

class YouTubeDownloaderEngine:
    """
    High-performance, multi-mode YouTube download, audio extraction,
    and direct CDN stream link extraction engine powered by yt-dlp.
    """

    @staticmethod
    def extract_playlist_metadata(
        urls: List[str],
        ui_queue: queue.Queue,
        cancel_event: threading.Event
    ) -> List[Dict[str, Any]]:
        """Extracts playlist or video metadata non-blockingly without downloading media."""
        if not YTDLP_AVAILABLE or not yt_dlp:
            ui_queue.put(("log", "[ERROR] yt-dlp library is not available. Please install via: pip install yt-dlp"))
            return []

        ui_queue.put(("status", "Parsing playlist and extracting video metadata..."))

        opts = {
            'quiet': True,
            'extract_flat': True,
            'ignoreerrors': True,
            'no_warnings': True,
            'logger': YtDlpLogger(ui_queue)
        }
        if FFMPEG_DIRECTORY:
            opts['ffmpeg_location'] = FFMPEG_DIRECTORY

        all_entries = []
        try:
            with yt_dlp.YoutubeDL(opts) as ydl:
                for target_url in urls:
                    if cancel_event.is_set():
                        ui_queue.put(("log", "[INFO] Metadata extraction aborted by user."))
                        break
                    ui_queue.put(("log", f"[INFO] Fetching playlist metadata: {target_url}"))
                    info = ydl.extract_info(target_url, download=False)
                    if not info:
                        ui_queue.put(("log", f"[WARNING] No metadata returned for: {target_url}"))
                        continue
                    entries = info.get('entries')
                    if entries is not None:
                        for e in entries:
                            if e:
                                all_entries.append(e)
                    else:
                        all_entries.append(info)
            if not all_entries:
                ui_queue.put(("log", "[WARNING] No videos found in the provided URL."))
            return all_entries
        except Exception as e:
            ui_queue.put(("log", f"[ERROR] Metadata extraction failed: {str(e)}"))
            return all_entries

    @staticmethod
    def fetch_thumbnail_bytes(url: str) -> Optional[bytes]:
        """Fetches raw thumbnail image bytes over HTTP."""
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=4) as resp:
                return resp.read()
        except Exception:
            return None

    @staticmethod
    def extract_direct_cdn_links_worker(
        urls: List[str],
        quality: str,
        ui_queue: queue.Queue,
        cancel_event: threading.Event
    ) -> List[Dict[str, Any]]:
        """
        Extracts direct GoogleVideo CDN streaming URLs based on requested quality.
        Pushes each found link into ui_queue for immediate in-dashboard display.
        """
        if not YTDLP_AVAILABLE or not yt_dlp:
            ui_queue.put(("log", "[ERROR] yt-dlp library is not available."))
            return []

        # Parse quality string (e.g. '1080p (Full HD)' -> '1080')
        short_q = quality.split(" ")[0].replace("p", "").strip().lower()

        ui_queue.put(("log", f"[INFO] Starting direct CDN stream extraction (Target Quality: {quality})..."))
        ui_queue.put(("status", "Extracting direct CDN stream URLs..."))

        flat_opts = {
            'quiet': True,
            'extract_flat': True,
            'ignoreerrors': True,
            'no_warnings': True,
            'logger': YtDlpLogger(ui_queue)
        }
        stream_opts = {
            'quiet': True,
            'ignoreerrors': True,
            'no_warnings': True,
            'logger': YtDlpLogger(ui_queue)
        }
        if FFMPEG_DIRECTORY:
            stream_opts['ffmpeg_location'] = FFMPEG_DIRECTORY

        extracted_links = []
        try:
            with yt_dlp.YoutubeDL(flat_opts) as ydl_flat, yt_dlp.YoutubeDL(stream_opts) as ydl_stream:
                video_targets = []
                for u in urls:
                    if cancel_event.is_set():
                        break
                    info = ydl_flat.extract_info(u, download=False)
                    if not info:
                        continue
                    if 'entries' in info and info['entries']:
                        for e in info['entries']:
                            if e:
                                v_url = e.get('url') or f"https://www.youtube.com/watch?v={e.get('id', '')}"
                                video_targets.append({
                                    "title": e.get('title', 'Unknown Title'),
                                    "url": v_url,
                                    "id": e.get('id', '')
                                })
                    else:
                        v_url = info.get('url') or u
                        video_targets.append({
                            "title": info.get('title', 'Unknown Title'),
                            "url": v_url,
                            "id": info.get('id', '')
                        })

                total_items = len(video_targets)
                ui_queue.put(("log", f"[INFO] Found {total_items} video item(s) to inspect for direct streaming links."))

                target_h = int(short_q) if short_q.isdigit() else 99999
                is_audio_only = "audio" in short_q

                for idx, v_item in enumerate(video_targets, 1):
                    if cancel_event.is_set():
                        ui_queue.put(("log", "[INFO] Direct CDN link extraction cancelled by user."))
                        break

                    pct = (idx / total_items) * 100.0 if total_items > 0 else 100.0
                    ui_queue.put(("progress", {
                        "percent": pct,
                        "downloaded": idx,
                        "total": total_items,
                        "speed": 0.0,
                        "eta": 0,
                        "filename": v_item["title"],
                        "status": "extracting"
                    }))
                    ui_queue.put(("status", f"Extracting stream URL {idx}/{total_items}..."))

                    try:
                        v_info = ydl_stream.extract_info(v_item["url"], download=False)
                        formats = v_info.get('formats') or []
                        candidates = [f for f in formats if f.get('url')]

                        if is_audio_only:
                            # Audio streams
                            a_cands = [f for f in candidates if f.get('vcodec') == 'none' and f.get('acodec') != 'none'] or candidates
                            a_cands.sort(key=lambda f: f.get('abr') or 0, reverse=True)
                            best = a_cands[0]
                            stream_url = best.get('url', '')
                            res_str = f"Audio ({best.get('ext', 'm4a')} - {int(best.get('abr') or 128)}k)"
                        elif candidates:
                            # Video streams
                            v_cands = [f for f in candidates if f.get('vcodec') != 'none'] or candidates
                            matching = [f for f in v_cands if (f.get('height') or 0) <= target_h] or v_cands
                            matching.sort(
                                key=lambda f: (f.get('vcodec') != 'none' and f.get('acodec') != 'none', f.get('height') or 0),
                                reverse=True
                            )
                            best = matching[0]
                            stream_url = best.get('url', '')
                            res_str = f"{best.get('height', 'Unknown')}p ({best.get('ext', '')})"
                        else:
                            stream_url = v_info.get('url', '')
                            res_str = "Auto"

                        if stream_url:
                            link_record = {
                                "idx": idx,
                                "selected": True,
                                "title": v_item["title"],
                                "resolution": res_str,
                                "url": stream_url,
                                "video_id": v_item["id"]
                            }
                            extracted_links.append(link_record)
                            ui_queue.put(("cdn_link_found", link_record))
                            ui_queue.put(("log", f"[STREAM {idx:03d}] {v_item['title'][:55]} -> {res_str}"))

                    except Exception as ex_item:
                        ui_queue.put(("log", f"[WARNING] Could not resolve direct link for item {idx}: {str(ex_item)}"))

            ui_queue.put(("cdn_links_ready", extracted_links))
            ui_queue.put(("log", f"[SUCCESS] Total direct CDN stream links extracted: {len(extracted_links)}"))
            return extracted_links
        except Exception as e:
            ui_queue.put(("log", f"[ERROR] Direct CDN link extraction failed: {str(e)}"))
            return extracted_links

    @staticmethod
    def execute_download(
        items_to_download: List[Dict[str, Any]],
        global_config: Dict[str, Any],
        ui_queue: queue.Queue,
        cancel_event: threading.Event
    ) -> bool:
        """Executes media download for selected videos with per-video quality resolution, strict byte telemetry, and concurrency."""
        if not YTDLP_AVAILABLE or not yt_dlp:
            ui_queue.put(("log", "[ERROR] yt-dlp library is missing. Aborting operation."))
            return False

        out_dir = global_config.get("out_dir", str(Path.home() / "Downloads" / "YouTube_Downloads"))
        Path(out_dir).mkdir(parents=True, exist_ok=True)

        numbering_index = global_config.get("numbering_index", True)
        embed_thumbnail = global_config.get("embed_thumbnail", True)
        embed_metadata = global_config.get("embed_metadata", True)
        write_subtitles = global_config.get("write_subtitles", False)
        rate_limit = global_config.get("rate_limit")
        concurrency = max(1, int(global_config.get("concurrency", 1)))

        total_count = len(items_to_download)
        ui_queue.put(("log", f"[START] Beginning batch download for {total_count} selected item(s) (Concurrency: {concurrency} parallel)..."))
        ui_queue.put(("status", f"Downloading batch (0/{total_count} completed)..."))

        # Initialize download task state for the Individual Progress Monitor
        init_tasks_data = []
        for it in items_to_download:
            init_tasks_data.append({
                "idx": it.get("idx", 0),
                "title": it.get("title", f"Video {it.get('idx', 0)}"),
                "quality": str(it.get("quality", global_config.get("resolution", "1080"))),
                "status": "queued",
                "percent": 0.0,
                "downloaded_bytes": 0,
                "total_bytes": it.get("bytes", 0),
                "speed": 0.0,
                "eta": 0
            })
        ui_queue.put(("init_tasks", init_tasks_data))

        progress_lock = threading.Lock()
        completed_items = 0
        active_items: Dict[int, Dict[str, Any]] = {}

        def make_progress_hook(item_idx: int, item_title: str):
            def progress_hook(d: Dict[str, Any]):
                if cancel_event.is_set():
                    raise yt_dlp.utils.DownloadCancelled("Download cancelled by user.")

                status = d.get('status')
                if status == 'downloading':
                    total_b = d.get('total_bytes') or d.get('total_bytes_estimate') or 0
                    downloaded_b = d.get('downloaded_bytes') or 0
                    pct = (downloaded_b / total_b * 100.0) if total_b > 0 else 0.0
                    speed_bps = d.get('speed') or 0.0
                    eta_sec = d.get('eta') or 0
                    filename = os.path.basename(d.get('filename', '')) or item_title

                    # Update individual task monitor
                    ui_queue.put(("item_progress", {
                        "idx": item_idx,
                        "title": item_title,
                        "status": "downloading",
                        "percent": pct,
                        "downloaded_bytes": downloaded_b,
                        "total_bytes": total_b,
                        "speed": speed_bps,
                        "eta": eta_sec
                    }))

                    with progress_lock:
                        active_items[item_idx] = {
                            "pct": pct,
                            "speed": speed_bps,
                            "downloaded": downloaded_b,
                            "total": total_b,
                            "filename": filename
                        }
                        total_speed = sum(v["speed"] for v in active_items.values())
                        total_dl = sum(v["downloaded"] for v in active_items.values())
                        total_est = sum(v["total"] for v in active_items.values())

                        if total_count == 1:
                            overall_pct = pct
                        else:
                            overall_pct = ((completed_items * 100.0) + sum(v["pct"] for v in active_items.values())) / (total_count * 100.0) * 100.0
                        overall_pct = min(100.0, max(0.0, overall_pct))

                        active_names = [v["filename"][:22] for v in active_items.values()]
                        active_label = ", ".join(active_names[:2])
                        if len(active_names) > 2:
                            active_label += f" +{len(active_names) - 2} more"

                    ui_queue.put(("progress", {
                        "percent": overall_pct,
                        "downloaded_bytes": total_dl,
                        "total_bytes": total_est,
                        "completed_items": completed_items,
                        "total_items": total_count,
                        "speed": total_speed,
                        "eta": eta_sec,
                        "filename": f"[{completed_items}/{total_count} done | {len(active_items)} active] {active_label}",
                        "status": "downloading"
                    }))
                elif status == 'finished':
                    filename = os.path.basename(d.get('filename', '')) or item_title
                    with progress_lock:
                        if item_idx in active_items:
                            active_items[item_idx]["pct"] = 100.0

                    ui_queue.put(("item_progress", {
                        "idx": item_idx,
                        "title": item_title,
                        "status": "postprocessing",
                        "percent": 100.0,
                        "downloaded_bytes": active_items.get(item_idx, {}).get("total", 0),
                        "total_bytes": active_items.get(item_idx, {}).get("total", 0),
                        "speed": 0.0,
                        "eta": 0
                    }))
            return progress_hook

        def download_single_item(idx: int, item: Dict[str, Any]) -> bool:
            nonlocal completed_items
            if cancel_event.is_set():
                return False

            item_url = item["url"]
            item_title = item.get("title", f"Video {idx}")
            item_quality = str(item.get("quality", global_config.get("resolution", "1080"))).lower().replace("p", "").strip()

            ui_queue.put(("log", f"[START {idx}/{total_count}] Downloading: {item_title[:60]} (Quality: {item_quality})"))
            ui_queue.put(("status", f"Downloading {item_title[:35]} ({completed_items}/{total_count} completed)..."))

            ui_queue.put(("item_progress", {
                "idx": idx,
                "title": item_title,
                "status": "downloading",
                "percent": 0.0,
                "downloaded_bytes": 0,
                "total_bytes": item.get("bytes", 0),
                "speed": 0.0,
                "eta": 0
            }))

            opts: Dict[str, Any] = {
                'logger': YtDlpLogger(ui_queue),
                'progress_hooks': [make_progress_hook(idx, item_title)],
                'ignoreerrors': True,
                'no_warnings': True,
                'windowsfilenames': True
            }

            if FFMPEG_DIRECTORY:
                opts['ffmpeg_location'] = FFMPEG_DIRECTORY

            if rate_limit:
                opts['ratelimit'] = rate_limit

            # Output template
            if numbering_index:
                opts['outtmpl'] = os.path.join(out_dir, f"{idx:02d} - %(title)s.%(ext)s")
            else:
                opts['outtmpl'] = os.path.join(out_dir, '%(title)s.%(ext)s')

            postprocessors = []

            # Check if this item is set to audio or video
            is_audio_mode = global_config.get("mode") == "audio_extractor"
            if item_quality in ("mp3", "m4a", "flac", "wav", "audio") or is_audio_mode:
                if item_quality in ("mp3", "m4a", "flac", "wav"):
                    codec = item_quality
                else:
                    codec = global_config.get("audio_codec", "mp3")
                preferred_quality = str(global_config.get("audio_quality", "320"))
                opts['format'] = 'bestaudio/best'
                postprocessors.append({
                    'key': 'FFmpegExtractAudio',
                    'preferredcodec': codec,
                    'preferredquality': preferred_quality,
                })
                if embed_thumbnail:
                    opts['writethumbnail'] = True
                    postprocessors.append({'key': 'FFmpegThumbnailsConvertor', 'format': 'jpg'})
                    postprocessors.append({'key': 'EmbedThumbnail'})
                if embed_metadata:
                    postprocessors.append({'key': 'FFmpegMetadata', 'add_chapters': True})
            else:
                # Video format
                container = global_config.get("container", "mp4").lower()
                if item_quality == "best":
                    opts['format'] = f"bestvideo+bestaudio/best"
                else:
                    opts['format'] = (
                        f"bestvideo[height<={item_quality}][ext={container}]+bestaudio[ext=m4a]/"
                        f"bestvideo[height<={item_quality}]+bestaudio/"
                        f"best[height<={item_quality}]/best"
                    )
                opts['merge_output_format'] = container

                if embed_thumbnail:
                    opts['writethumbnail'] = True
                    postprocessors.append({'key': 'FFmpegThumbnailsConvertor', 'format': 'jpg'})
                    postprocessors.append({'key': 'EmbedThumbnail'})
                if embed_metadata:
                    postprocessors.append({'key': 'FFmpegMetadata', 'add_chapters': True})
                if write_subtitles:
                    opts['writesubtitles'] = True
                    opts['subtitleslangs'] = ['en', 'all']

            if postprocessors:
                opts['postprocessors'] = postprocessors

            try:
                with yt_dlp.YoutubeDL(opts) as ydl:
                    ydl.download([item_url])

                with progress_lock:
                    completed_items += 1
                    active_items.pop(idx, None)
                    c = completed_items
                    overall_pct = (c / total_count) * 100.0

                ui_queue.put(("item_progress", {
                    "idx": idx,
                    "title": item_title,
                    "status": "completed",
                    "percent": 100.0,
                    "downloaded_bytes": item.get("bytes", 0),
                    "total_bytes": item.get("bytes", 0),
                    "speed": 0.0,
                    "eta": 0
                }))

                ui_queue.put(("progress", {
                    "percent": overall_pct,
                    "downloaded_bytes": 0,
                    "total_bytes": 0,
                    "completed_items": c,
                    "total_items": total_count,
                    "speed": 0.0,
                    "eta": 0,
                    "filename": f"[{c}/{total_count}] {item_title[:45]} finished",
                    "status": "finished"
                }))
                ui_queue.put(("log", f"[COMPLETED {c}/{total_count}] {item_title[:60]} finished."))
                ui_queue.put(("status", f"Completed {c}/{total_count} videos ({total_count - c} remaining)..."))
                return True
            except yt_dlp.utils.DownloadCancelled:
                with progress_lock:
                    active_items.pop(idx, None)
                ui_queue.put(("item_progress", {
                    "idx": idx,
                    "title": item_title,
                    "status": "cancelled",
                    "percent": 0.0
                }))
                ui_queue.put(("log", f"[CANCELLED] Video {idx} cancelled."))
                return False
            except Exception as e:
                with progress_lock:
                    active_items.pop(idx, None)
                ui_queue.put(("item_progress", {
                    "idx": idx,
                    "title": item_title,
                    "status": "error",
                    "percent": 0.0,
                    "error": str(e)
                }))
                ui_queue.put(("log", f"[ERROR] Failed downloading item {idx}: {str(e)}"))
                return False

        if concurrency == 1 or total_count <= 1:
            ui_queue.put(("log", "[EXECUTION] Running sequential download queue (1 video at a time)..."))
            for idx, item in enumerate(items_to_download, 1):
                if cancel_event.is_set():
                    ui_queue.put(("log", "[INFO] Batch download cancelled by user."))
                    return False
                download_single_item(idx, item)
        else:
            ui_queue.put(("log", f"[CONCURRENCY] Running parallel download queue ({concurrency} videos simultaneously)..."))
            with concurrent.futures.ThreadPoolExecutor(max_workers=concurrency) as executor:
                futures = {
                    executor.submit(download_single_item, idx, item): idx
                    for idx, item in enumerate(items_to_download, 1)
                }
                for future in concurrent.futures.as_completed(futures):
                    if cancel_event.is_set():
                        for f in futures:
                            f.cancel()
                        break

        ui_queue.put(("log", f"[SUCCESS] Batch download completed. Total items processed: {total_count} (Completed: {completed_items})"))
        ui_queue.put(("status", f"Batch download completed ({completed_items}/{total_count} successful)."))
        return completed_items > 0


# ==============================================================================
# DESKTOP GUI APPLICATION CLASS
# ==============================================================================

class YouTubeDownloaderGUI:
    """
    Enterprise Tkinter Desktop GUI for YouTube media processing.
    Fully thread-safe with Python 3.13 via queue.Queue.
    """

    COLOR_BG = "#0B1120"
    COLOR_CARD = "#162032"
    COLOR_CARD_BORDER = "#1E293B"
    COLOR_CARD_HOVER = "#243247"
    COLOR_PRIMARY = "#0284C7"
    COLOR_PRIMARY_HOVER = "#0369A1"
    COLOR_SUCCESS = "#059669"
    COLOR_SUCCESS_HOVER = "#047857"
    COLOR_DANGER = "#DC2626"
    COLOR_DANGER_HOVER = "#B91C1C"
    COLOR_WARNING = "#D97706"
    COLOR_TEXT = "#F8FAFC"
    COLOR_TEXT_SECONDARY = "#94A3B8"
    COLOR_TEXT_MUTED = "#64748B"
    COLOR_INPUT_BG = "#0F172A"
    COLOR_ROW_CHECKED = "#132D4B"       # Highlighted background for checked items
    COLOR_ROW_UNCHECKED = "#0F172A"     # Standard dark background for unchecked
    COLOR_LOG_BG = "#070B14"
    COLOR_LOG_TEXT = "#38BDF8"

    MODES = [
        ("playlist", "Playlist Downloader", "Inspect playlist items with thumbnails, configure individual video qualities, and batch download"),
        ("single_video", "Single Video Downloader", "Fast on-demand single video stream download with resolution presets"),
        ("audio_extractor", "Batch Audio Extractor", "Extract pristine audio streams to MP3, M4A, FLAC, or WAV"),
        ("picker", "Metadata Inspector & Picker", "Inspect playlist items in an interactive table and selectively download"),
        ("direct_links", "Direct CDN Link Exporter", "Extract raw GoogleVideo direct streaming links for IDM & aria2")
    ]
    MODE_TITLES = {k: title for k, title, _ in MODES}

    CONCURRENT_OPTIONS = [
        "1 (Sequential)",
        "2 Videos Together",
        "3 Videos Together",
        "4 Videos Together",
        "5 Videos Together",
        "8 Videos Together",
        "10 Videos Together"
    ]

    QUALITY_OPTIONS = [
        "1080p (Full HD)",
        "720p (HD)",
        "480p (SD)",
        "360p",
        "2160p (4K)",
        "1440p (2K)",
        "Audio (MP3 320k)",
        "Audio (M4A)"
    ]

    CDN_QUALITY_OPTIONS = [
        "1080p (Full HD)",
        "720p (HD)",
        "480p (SD)",
        "360p",
        "2160p (4K)",
        "1440p (2K)",
        "Best Available",
        "Audio Only (m4a/opus)"
    ]

    def __init__(self, parent=None):
        self.root = tk.Tk() if parent is None else tk.Toplevel(parent)
        self.root.title("YouTube Media Suite & Playlist Toolkit Pro -- Developed by mohatamim")

        # Responsive window sizing
        sw = self.root.winfo_screenwidth()
        sh = self.root.winfo_screenheight()
        target_w = min(1280, max(1060, sw - 60))
        target_h = min(780, max(660, sh - 90))
        self.root.geometry(f"{target_w}x{target_h}")
        self.root.minsize(1040, 620)
        self.root.configure(bg=self.COLOR_BG)

        self.current_mode = "playlist"
        self.is_running = False
        self.worker_thread: Optional[threading.Thread] = None
        self.cancel_event = threading.Event()
        self.ui_queue: queue.Queue = queue.Queue()

        # Data stores
        self.inspected_entries: List[Dict[str, Any]] = []
        self.cdn_links: List[Dict[str, Any]] = []
        self.thumbnail_cache: Dict[int, Any] = {}
        self.placeholder_img = None
        self._init_placeholder_image()

        # Active selected video in inspector
        self.active_entry_idx: Optional[int] = None

        # Form variables (Default URL cleared as requested)
        self.url_var = tk.StringVar(value="")
        default_out = str(Path.home() / "Downloads" / "YouTube_Downloads")
        self.out_dir_var = tk.StringVar(value=default_out)
        self.open_dest_var = tk.BooleanVar(value=True)

        # Global quality & format options
        self.res_var = tk.StringVar(value="1080")
        self.container_var = tk.StringVar(value="mp4")
        self.audio_codec_var = tk.StringVar(value="mp3")
        self.audio_quality_var = tk.StringVar(value="320")
        self.speed_limit_var = tk.StringVar(value="Unlimited")
        self.concurrent_downloads_var = tk.StringVar(value="3 Videos Together")
        self.download_tasks_state = {}
        self.progress_modal = None
        self.progress_modal_items = {}
        self.media_player_modal = None

        # Active inspector panel variables
        self.active_title_var = tk.StringVar(value="No Video Selected")
        self.active_info_var = tk.StringVar(value="Select any row from the playlist table to adjust individual settings.")
        self.active_quality_var = tk.StringVar(value="1080p (Full HD)")
        self.batch_apply_quality_var = tk.StringVar(value="1080p (Full HD)")

        # CDN Tab specific quality variable
        self.cdn_quality_var = tk.StringVar(value="1080p (Full HD)")

        # Filter & Search variable
        self.filter_var = tk.StringVar(value="")
        self.filter_var.trace_add("write", lambda *args: self._on_filter_changed())

        # Playlist filters
        self.playlist_start_var = tk.StringVar(value="")
        self.playlist_end_var = tk.StringVar(value="")
        self.numbering_var = tk.BooleanVar(value=True)

        # Extra flags
        self.thumbnail_var = tk.BooleanVar(value=True)
        self.metadata_var = tk.BooleanVar(value=True)
        self.subtitles_var = tk.BooleanVar(value=False)

        # Telemetry variables
        self.progress_var = tk.DoubleVar(value=0.0)
        self.status_text_var = tk.StringVar(value="Ready. Paste a playlist or video link, then click 'FETCH & LOAD PLAYLIST VIDEOS'.")
        self.speed_text_var = tk.StringVar(value="Speed: 0.0 MB/s | ETA: 00:00 | Size: 0.0 MB / 0.0 MB")
        self.active_item_var = tk.StringVar(value="Active Item: Ready")
        self.auto_scroll_var = tk.BooleanVar(value=True)

        # Build interface
        self._setup_styles()
        self._build_ui()
        self._apply_mode_ui_state()

        # Bind keyboard shortcuts
        self._bind_shortcuts()

        # Start UI queue polling loop (30ms interval)
        self.root.after(30, self._poll_queue)

        # Windnd support if available
        if WINDND_AVAILABLE:
            try:
                windnd.hook_dropfiles(self.root, func=self._on_windnd_drop)
            except Exception:
                pass

    def _init_placeholder_image(self):
        """Generates a default dark thumbnail placeholder."""
        if PILLOW_AVAILABLE:
            img = Image.new('RGB', (64, 36), color='#1E293B')
            draw = ImageDraw.Draw(img)
            draw.polygon([(26, 11), (26, 25), (40, 18)], fill='#38BDF8')
            self.placeholder_img = ImageTk.PhotoImage(img)

    def _setup_styles(self):
        """Initializes dark ttk styles for treeviews, progress bars, and tabs."""
        style = ttk.Style()
        try:
            style.theme_use('clam')
        except Exception:
            pass

        # Progress bar
        style.configure(
            "TProgressbar",
            troughcolor=self.COLOR_CARD,
            background=self.COLOR_PRIMARY,
            bordercolor=self.COLOR_CARD_BORDER,
            lightcolor=self.COLOR_PRIMARY_HOVER,
            darkcolor=self.COLOR_PRIMARY
        )

        # Notebook tabs styling
        style.configure(
            "TNotebook",
            background=self.COLOR_BG,
            borderwidth=0
        )
        style.configure(
            "TNotebook.Tab",
            background=self.COLOR_CARD,
            foreground=self.COLOR_TEXT_SECONDARY,
            padding=[16, 7],
            font=("Segoe UI", 9, "bold"),
            borderwidth=0
        )
        style.map(
            "TNotebook.Tab",
            background=[("selected", self.COLOR_PRIMARY), ("active", self.COLOR_CARD_HOVER)],
            foreground=[("selected", "#FFFFFF"), ("active", self.COLOR_TEXT)]
        )

        # Combobox styling
        style.configure(
            "TCombobox",
            background=self.COLOR_CARD,
            foreground="#F8FAFC",
            fieldbackground=self.COLOR_INPUT_BG,
            bordercolor=self.COLOR_CARD_BORDER,
            darkcolor=self.COLOR_CARD_BORDER,
            lightcolor=self.COLOR_CARD_BORDER,
            arrowcolor="#38BDF8",
            arrowsize=14
        )
        style.map(
            "TCombobox",
            fieldbackground=[("readonly", self.COLOR_INPUT_BG)],
            selectbackground=[("readonly", self.COLOR_PRIMARY)],
            selectforeground=[("readonly", "#FFFFFF")]
        )

        # Treeview styling with tall rowheight for thumbnails
        style.configure(
            "Treeview",
            background=self.COLOR_INPUT_BG,
            foreground=self.COLOR_TEXT,
            fieldbackground=self.COLOR_INPUT_BG,
            bordercolor=self.COLOR_CARD_BORDER,
            borderwidth=1,
            rowheight=42,
            font=("Segoe UI", 9)
        )
        style.configure(
            "Treeview.Heading",
            background=self.COLOR_CARD,
            foreground=self.COLOR_TEXT_SECONDARY,
            relief="flat",
            font=("Segoe UI", 9, "bold")
        )
        style.map(
            "Treeview.Heading",
            background=[('active', self.COLOR_CARD_HOVER)],
            foreground=[('active', self.COLOR_TEXT)]
        )
        style.map(
            "Treeview",
            background=[('selected', self.COLOR_PRIMARY)],
            foreground=[('selected', "#FFFFFF")]
        )

    def _build_ui(self):
        """Constructs the high-density glassmorphism dashboard layout."""
        # Top Header Banner
        header_frame = tk.Frame(self.root, bg=self.COLOR_CARD, height=66, highlightthickness=1, highlightbackground=self.COLOR_CARD_BORDER)
        header_frame.pack(fill="x", side="top", padx=12, pady=(8, 4))
        header_frame.pack_propagate(False)

        title_sub_frame = tk.Frame(header_frame, bg=self.COLOR_CARD)
        title_sub_frame.pack(side="left", padx=16, pady=8)

        lbl_title = tk.Label(
            title_sub_frame,
            text="YOUTUBE MEDIA SUITE & PLAYLIST TOOLKIT PRO",
            font=("Segoe UI", 13, "bold"),
            fg=self.COLOR_TEXT,
            bg=self.COLOR_CARD
        )
        lbl_title.pack(anchor="w")

        lbl_sub = tk.Label(
            title_sub_frame,
            text="Multi-Engine Downloader, Audio Extractor, Metadata Inspector & CDN Stream Link Exporter",
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT_SECONDARY,
            bg=self.COLOR_CARD
        )
        lbl_sub.pack(anchor="w")

        # Top Right Badges & Shortcut Trigger
        badge_frame = tk.Frame(header_frame, bg=self.COLOR_CARD)
        badge_frame.pack(side="right", padx=16, pady=8)

        author_badge = tk.Label(
            badge_frame,
            text="Developed by: mohatamim",
            font=("Segoe UI", 8, "bold"),
            fg="#38BDF8",
            bg="#0F172A",
            padx=10,
            pady=4,
            relief="solid",
            bd=1
        )
        author_badge.pack(side="right", padx=(8, 0))

        btn_shortcuts = tk.Button(
            badge_frame,
            text="Shortcuts (F1)",
            font=("Segoe UI", 8, "bold"),
            fg=self.COLOR_TEXT,
            bg=self.COLOR_CARD_BORDER,
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_PRIMARY,
            cursor="hand2",
            bd=0,
            padx=10,
            pady=4,
            command=self._show_shortcuts_modal
        )
        btn_shortcuts.pack(side="right")

        # Mode Selection Bar
        self.mode_frame = tk.Frame(self.root, bg=self.COLOR_BG)
        self.mode_frame.pack(fill="x", side="top", padx=12, pady=4)

        self.mode_buttons = {}
        for idx, (m_key, m_title, m_desc) in enumerate(self.MODES, 1):
            btn = tk.Button(
                self.mode_frame,
                text=f"Alt+{idx}: {m_title}",
                font=("Segoe UI", 9, "bold"),
                fg=self.COLOR_TEXT_SECONDARY,
                bg=self.COLOR_CARD,
                activeforeground=self.COLOR_TEXT,
                activebackground=self.COLOR_PRIMARY,
                cursor="hand2",
                bd=0,
                padx=10,
                pady=5,
                command=lambda k=m_key: self._switch_mode(k)
            )
            btn.pack(side="left", padx=(0, 5), fill="x", expand=True)
            self.mode_buttons[m_key] = btn

        # Main Body Splitter: Left Configuration Panel (440px), Right Tabbed Workspace (Fill)
        main_body = tk.Frame(self.root, bg=self.COLOR_BG)
        main_body.pack(fill="both", expand=True, padx=12, pady=4)

        left_panel = tk.Frame(main_body, bg=self.COLOR_CARD, width=440, highlightthickness=1, highlightbackground=self.COLOR_CARD_BORDER)
        left_panel.pack(side="left", fill="y", padx=(0, 6), pady=0)
        left_panel.pack_propagate(False)

        right_panel = tk.Frame(main_body, bg=self.COLOR_BG)
        right_panel.pack(side="right", fill="both", expand=True, padx=(4, 0), pady=0)

        # ----------------------------------------------------------------------
        # LEFT PANEL: URL, SETTINGS, EXTRAS, ACTION BUTTONS
        # ----------------------------------------------------------------------
        # Section 1: URL Input
        url_sec = tk.LabelFrame(
            left_panel,
            text=" Target URL(s) / Playlist Identifier ",
            font=("Segoe UI", 9, "bold"),
            fg=self.COLOR_TEXT,
            bg=self.COLOR_CARD,
            bd=1,
            relief="solid"
        )
        url_sec.pack(fill="x", padx=10, pady=(8, 4))

        url_input_row = tk.Frame(url_sec, bg=self.COLOR_CARD)
        url_input_row.pack(fill="x", padx=8, pady=5)

        self.ent_url = tk.Entry(
            url_input_row,
            textvariable=self.url_var,
            font=("Segoe UI", 9),
            bg=self.COLOR_INPUT_BG,
            fg=self.COLOR_TEXT,
            insertbackground="#FFFFFF",
            relief="solid",
            bd=1
        )
        self.ent_url.pack(side="left", fill="x", expand=True, ipady=4)

        btn_paste = tk.Button(
            url_input_row,
            text="Paste",
            font=("Segoe UI", 8, "bold"),
            fg=self.COLOR_TEXT,
            bg=self.COLOR_PRIMARY,
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_PRIMARY_HOVER,
            cursor="hand2",
            bd=0,
            padx=8,
            pady=3,
            command=self._paste_url
        )
        btn_paste.pack(side="left", padx=(4, 0))

        btn_clear_url = tk.Button(
            url_input_row,
            text="X",
            font=("Segoe UI", 8, "bold"),
            fg=self.COLOR_TEXT_SECONDARY,
            bg="#334155",
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_DANGER,
            cursor="hand2",
            bd=0,
            padx=6,
            pady=3,
            command=lambda: self.url_var.set("")
        )
        btn_clear_url.pack(side="left", padx=(3, 0))

        btn_multi = tk.Button(
            url_input_row,
            text="+ Multi",
            font=("Segoe UI", 8, "bold"),
            fg=self.COLOR_TEXT,
            bg="#334155",
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_CARD_HOVER,
            cursor="hand2",
            bd=0,
            padx=6,
            pady=3,
            command=self._show_multi_urls_dialog
        )
        btn_multi.pack(side="left", padx=(3, 0))

        self.btn_inspect_playlist = tk.Button(
            url_sec,
            text="FETCH & LOAD PLAYLIST VIDEOS",
            font=("Segoe UI", 9, "bold"),
            fg="#FFFFFF",
            bg="#0284C7",
            activeforeground="#FFFFFF",
            activebackground="#0369A1",
            cursor="hand2",
            bd=0,
            pady=5,
            command=self._fetch_metadata_inspect
        )
        self.btn_inspect_playlist.pack(fill="x", padx=8, pady=(2, 6))

        # Section 2: Destination Output Folder
        out_sec = tk.LabelFrame(
            left_panel,
            text=" Output Destination ",
            font=("Segoe UI", 9, "bold"),
            fg=self.COLOR_TEXT,
            bg=self.COLOR_CARD,
            bd=1,
            relief="solid"
        )
        out_sec.pack(fill="x", padx=10, pady=4)

        out_row = tk.Frame(out_sec, bg=self.COLOR_CARD)
        out_row.pack(fill="x", padx=8, pady=5)

        ent_out = tk.Entry(
            out_row,
            textvariable=self.out_dir_var,
            font=("Segoe UI", 9),
            bg=self.COLOR_INPUT_BG,
            fg=self.COLOR_TEXT,
            insertbackground="#FFFFFF",
            relief="solid",
            bd=1
        )
        ent_out.pack(side="left", fill="x", expand=True, ipady=4)

        btn_browse = tk.Button(
            out_row,
            text="Browse",
            font=("Segoe UI", 8, "bold"),
            fg=self.COLOR_TEXT,
            bg=self.COLOR_PRIMARY,
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_PRIMARY_HOVER,
            cursor="hand2",
            bd=0,
            padx=8,
            pady=3,
            command=self._browse_output_dir
        )
        btn_browse.pack(side="left", padx=(4, 2))

        btn_open_dir = tk.Button(
            out_row,
            text="Open",
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT,
            bg="#334155",
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_CARD_HOVER,
            cursor="hand2",
            bd=0,
            padx=6,
            pady=3,
            command=self._open_output_dir
        )
        btn_open_dir.pack(side="left")

        chk_open_dest = tk.Checkbutton(
            out_sec,
            text="Open destination directory upon completion",
            variable=self.open_dest_var,
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT_SECONDARY,
            bg=self.COLOR_CARD,
            activebackground=self.COLOR_CARD,
            activeforeground=self.COLOR_TEXT,
            selectcolor=self.COLOR_INPUT_BG,
            cursor="hand2"
        )
        chk_open_dest.pack(anchor="w", padx=8, pady=(0, 4))

        # Section 3: Dynamic Parameters & Options Panel
        self.params_sec = tk.LabelFrame(
            left_panel,
            text=" Default Encoding & Quality Presets ",
            font=("Segoe UI", 9, "bold"),
            fg=self.COLOR_TEXT,
            bg=self.COLOR_CARD,
            bd=1,
            relief="solid"
        )
        self.params_sec.pack(fill="x", padx=10, pady=4)

        # Video Quality & Format Controls Frame
        self.video_opts_frame = tk.Frame(self.params_sec, bg=self.COLOR_CARD)
        self.video_opts_frame.pack(fill="x", padx=8, pady=3)

        lbl_res = tk.Label(self.video_opts_frame, text="Default Quality:", font=("Segoe UI", 8, "bold"), fg=self.COLOR_TEXT_SECONDARY, bg=self.COLOR_CARD)
        lbl_res.grid(row=0, column=0, sticky="w", pady=3)

        cbo_res = ttk.Combobox(
            self.video_opts_frame,
            textvariable=self.res_var,
            values=["best", "2160", "1440", "1080", "720", "480", "360"],
            state="readonly",
            width=8,
            cursor="hand2"
        )
        cbo_res.grid(row=0, column=1, sticky="w", padx=5, pady=3)

        lbl_cont = tk.Label(self.video_opts_frame, text="Format:", font=("Segoe UI", 8, "bold"), fg=self.COLOR_TEXT_SECONDARY, bg=self.COLOR_CARD)
        lbl_cont.grid(row=0, column=2, sticky="w", padx=(8, 0), pady=3)

        cbo_cont = ttk.Combobox(
            self.video_opts_frame,
            textvariable=self.container_var,
            values=["mp4", "mkv", "webm"],
            state="readonly",
            width=7,
            cursor="hand2"
        )
        cbo_cont.grid(row=0, column=3, sticky="w", padx=5, pady=3)

        # Audio Quality & Format Controls Frame
        self.audio_opts_frame = tk.Frame(self.params_sec, bg=self.COLOR_CARD)
        self.audio_opts_frame.pack(fill="x", padx=8, pady=3)

        lbl_acodec = tk.Label(self.audio_opts_frame, text="Audio Format:", font=("Segoe UI", 8, "bold"), fg=self.COLOR_TEXT_SECONDARY, bg=self.COLOR_CARD)
        lbl_acodec.grid(row=0, column=0, sticky="w", pady=3)

        cbo_acodec = ttk.Combobox(
            self.audio_opts_frame,
            textvariable=self.audio_codec_var,
            values=["mp3", "m4a", "flac", "wav"],
            state="readonly",
            width=7,
            cursor="hand2"
        )
        cbo_acodec.grid(row=0, column=1, sticky="w", padx=5, pady=3)

        lbl_aqual = tk.Label(self.audio_opts_frame, text="Bitrate:", font=("Segoe UI", 8, "bold"), fg=self.COLOR_TEXT_SECONDARY, bg=self.COLOR_CARD)
        lbl_aqual.grid(row=0, column=2, sticky="w", padx=(8, 0), pady=3)

        cbo_aqual = ttk.Combobox(
            self.audio_opts_frame,
            textvariable=self.audio_quality_var,
            values=["320", "256", "192", "128", "0"],
            state="readonly",
            width=7,
            cursor="hand2"
        )
        cbo_aqual.grid(row=0, column=3, sticky="w", padx=5, pady=3)

        # Playlist Range Options Frame
        self.range_opts_frame = tk.Frame(self.params_sec, bg=self.COLOR_CARD)
        self.range_opts_frame.pack(fill="x", padx=8, pady=3)

        lbl_p_range = tk.Label(self.range_opts_frame, text="Range (Start - End):", font=("Segoe UI", 8, "bold"), fg=self.COLOR_TEXT_SECONDARY, bg=self.COLOR_CARD)
        lbl_p_range.grid(row=0, column=0, sticky="w", pady=3)

        ent_start = tk.Entry(self.range_opts_frame, textvariable=self.playlist_start_var, width=5, bg=self.COLOR_INPUT_BG, fg=self.COLOR_TEXT, relief="solid", bd=1)
        ent_start.grid(row=0, column=1, padx=(6, 3), pady=3)

        lbl_dash = tk.Label(self.range_opts_frame, text="to", font=("Segoe UI", 8), fg=self.COLOR_TEXT_MUTED, bg=self.COLOR_CARD)
        lbl_dash.grid(row=0, column=2, pady=3)

        ent_end = tk.Entry(self.range_opts_frame, textvariable=self.playlist_end_var, width=5, bg=self.COLOR_INPUT_BG, fg=self.COLOR_TEXT, relief="solid", bd=1)
        ent_end.grid(row=0, column=3, padx=(3, 0), pady=3)

        # Performance, Bandwidth & Concurrency Controls
        perf_row = tk.Frame(self.params_sec, bg=self.COLOR_CARD)
        perf_row.pack(fill="x", padx=8, pady=3)

        lbl_concurrency = tk.Label(perf_row, text="Download Together:", font=("Segoe UI", 8, "bold"), fg=self.COLOR_TEXT_SECONDARY, bg=self.COLOR_CARD)
        lbl_concurrency.grid(row=0, column=0, sticky="w", pady=2)

        cbo_concurrency = ttk.Combobox(
            perf_row,
            textvariable=self.concurrent_downloads_var,
            values=self.CONCURRENT_OPTIONS,
            state="readonly",
            width=14,
            cursor="hand2"
        )
        cbo_concurrency.grid(row=0, column=1, sticky="w", padx=(4, 10), pady=2)

        lbl_throttle = tk.Label(perf_row, text="Speed Limit:", font=("Segoe UI", 8, "bold"), fg=self.COLOR_TEXT_SECONDARY, bg=self.COLOR_CARD)
        lbl_throttle.grid(row=0, column=2, sticky="w", pady=2)

        cbo_throttle = ttk.Combobox(
            perf_row,
            textvariable=self.speed_limit_var,
            values=["Unlimited", "10 MB/s", "5 MB/s", "2 MB/s", "1 MB/s"],
            state="readonly",
            width=10,
            cursor="hand2"
        )
        cbo_throttle.grid(row=0, column=3, sticky="w", padx=4, pady=2)

        # Section 4: Extra Metadata Flags
        extras_sec = tk.LabelFrame(
            left_panel,
            text=" Metadata & Post-Processing Options ",
            font=("Segoe UI", 9, "bold"),
            fg=self.COLOR_TEXT,
            bg=self.COLOR_CARD,
            bd=1,
            relief="solid"
        )
        extras_sec.pack(fill="x", padx=10, pady=4)

        chk_num = tk.Checkbutton(
            extras_sec,
            text="Prefix Playlist Index to Filename (01 - Title.mp4)",
            variable=self.numbering_var,
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT,
            bg=self.COLOR_CARD,
            activebackground=self.COLOR_CARD,
            activeforeground=self.COLOR_TEXT,
            selectcolor=self.COLOR_INPUT_BG,
            cursor="hand2"
        )
        chk_num.pack(anchor="w", padx=8, pady=1)

        chk_thumb = tk.Checkbutton(
            extras_sec,
            text="Embed High-Resolution Thumbnail into File",
            variable=self.thumbnail_var,
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT,
            bg=self.COLOR_CARD,
            activebackground=self.COLOR_CARD,
            activeforeground=self.COLOR_TEXT,
            selectcolor=self.COLOR_INPUT_BG,
            cursor="hand2"
        )
        chk_thumb.pack(anchor="w", padx=8, pady=1)

        chk_meta = tk.Checkbutton(
            extras_sec,
            text="Embed Metadata Tags, Chapters & Description",
            variable=self.metadata_var,
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT,
            bg=self.COLOR_CARD,
            activebackground=self.COLOR_CARD,
            activeforeground=self.COLOR_TEXT,
            selectcolor=self.COLOR_INPUT_BG,
            cursor="hand2"
        )
        chk_meta.pack(anchor="w", padx=8, pady=1)

        chk_subs = tk.Checkbutton(
            extras_sec,
            text="Download English / Auto Subtitles (.srt / .vtt)",
            variable=self.subtitles_var,
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT,
            bg=self.COLOR_CARD,
            activebackground=self.COLOR_CARD,
            activeforeground=self.COLOR_TEXT,
            selectcolor=self.COLOR_INPUT_BG,
            cursor="hand2"
        )
        chk_subs.pack(anchor="w", padx=8, pady=1)

        # Section 5: Primary Action Buttons
        action_sec = tk.Frame(left_panel, bg=self.COLOR_CARD)
        action_sec.pack(fill="x", side="bottom", padx=10, pady=(4, 8))

        self.btn_execute = tk.Button(
            action_sec,
            text="DOWNLOAD PLAYLIST (F5)",
            font=("Segoe UI", 10, "bold"),
            fg="#FFFFFF",
            bg=self.COLOR_SUCCESS,
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_SUCCESS_HOVER,
            cursor="hand2",
            bd=0,
            pady=8,
            command=self._start_execution
        )
        self.btn_execute.pack(fill="x", pady=(0, 4))

        cancel_row = tk.Frame(action_sec, bg=self.COLOR_CARD)
        cancel_row.pack(fill="x")

        self.btn_cancel = tk.Button(
            cancel_row,
            text="Cancel Execution (Esc)",
            font=("Segoe UI", 8, "bold"),
            fg="#FFFFFF",
            bg=self.COLOR_DANGER,
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_DANGER_HOVER,
            cursor="hand2",
            bd=0,
            pady=5,
            state="disabled",
            command=self._cancel_execution
        )
        self.btn_cancel.pack(side="left", fill="x", expand=True, padx=(0, 4))

        self.btn_clear_queue = tk.Button(
            cancel_row,
            text="Clear Form",
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT,
            bg="#334155",
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_CARD_HOVER,
            cursor="hand2",
            bd=0,
            pady=5,
            command=self._clear_form
        )
        self.btn_clear_queue.pack(side="right", fill="x", expand=True, padx=(4, 0))

        # ----------------------------------------------------------------------
        # RIGHT PANEL: GUARANTEED VISIBLE FOOTER (LOG + TELEMETRY) & TOP TABS
        # ----------------------------------------------------------------------
        # 1. Real-Time Output Console Log Section (Pack FIRST at bottom to guarantee visibility!)
        log_sec = tk.LabelFrame(
            right_panel,
            text=" Live Telemetry & Process Execution Log ",
            font=("Segoe UI", 9, "bold"),
            fg=self.COLOR_TEXT,
            bg=self.COLOR_CARD,
            bd=1,
            relief="solid",
            height=140
        )
        log_sec.pack(side="bottom", fill="x", pady=(3, 0))
        log_sec.pack_propagate(False)

        log_toolbar = tk.Frame(log_sec, bg=self.COLOR_CARD)
        log_toolbar.pack(fill="x", padx=8, pady=(2, 3))

        chk_scroll = tk.Checkbutton(
            log_toolbar,
            text="Auto-scroll",
            variable=self.auto_scroll_var,
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT_SECONDARY,
            bg=self.COLOR_CARD,
            activebackground=self.COLOR_CARD,
            activeforeground=self.COLOR_TEXT,
            selectcolor=self.COLOR_INPUT_BG,
            cursor="hand2"
        )
        chk_scroll.pack(side="left")

        btn_clear_log = tk.Button(
            log_toolbar,
            text="Clear Log",
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT,
            bg="#1E293B",
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_PRIMARY,
            cursor="hand2",
            bd=0,
            padx=8,
            pady=2,
            command=self._clear_log
        )
        btn_clear_log.pack(side="right", padx=(4, 0))

        btn_copy_log = tk.Button(
            log_toolbar,
            text="Copy Log",
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT,
            bg="#1E293B",
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_PRIMARY,
            cursor="hand2",
            bd=0,
            padx=8,
            pady=2,
            command=self._copy_log
        )
        btn_copy_log.pack(side="right")

        log_frame = tk.Frame(log_sec, bg=self.COLOR_LOG_BG)
        log_frame.pack(fill="both", expand=True, padx=8, pady=(0, 4))

        self.txt_log = tk.Text(
            log_frame,
            bg=self.COLOR_LOG_BG,
            fg=self.COLOR_LOG_TEXT,
            font=("Consolas", 9),
            relief="flat",
            wrap="word",
            state="disabled",
            bd=0
        )
        log_scroll = ttk.Scrollbar(log_frame, orient="vertical", command=self.txt_log.yview)
        self.txt_log.configure(yscrollcommand=log_scroll.set)

        self.txt_log.pack(side="left", fill="both", expand=True)
        log_scroll.pack(side="right", fill="y")

        # 2. Telemetry & Status Bar Section (Pack SECOND at bottom, directly above Log)
        telemetry_frame = tk.Frame(right_panel, bg=self.COLOR_CARD, highlightthickness=1, highlightbackground=self.COLOR_CARD_BORDER)
        telemetry_frame.pack(side="bottom", fill="x", pady=(3, 0))

        telemetry_inner = tk.Frame(telemetry_frame, bg=self.COLOR_CARD)
        telemetry_inner.pack(fill="x", padx=10, pady=5)

        status_row = tk.Frame(telemetry_inner, bg=self.COLOR_CARD)
        status_row.pack(fill="x", pady=(0, 2))

        lbl_status = tk.Label(
            status_row,
            textvariable=self.status_text_var,
            font=("Segoe UI", 9, "bold"),
            fg=self.COLOR_TEXT,
            bg=self.COLOR_CARD
        )
        lbl_status.pack(side="left")

        btn_open_monitor = tk.Button(
            status_row,
            text="Task Monitor (Click Progress Bar)",
            font=("Segoe UI", 8, "bold"),
            fg="#38BDF8",
            bg="#0F172A",
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_PRIMARY,
            cursor="hand2",
            bd=1,
            relief="solid",
            padx=8,
            pady=1,
            command=self._open_individual_progress_modal
        )
        btn_open_monitor.pack(side="right", padx=(8, 0))

        lbl_speed = tk.Label(
            status_row,
            textvariable=self.speed_text_var,
            font=("Segoe UI", 8),
            fg="#38BDF8",
            bg=self.COLOR_CARD
        )
        lbl_speed.pack(side="right")

        self.progressbar = ttk.Progressbar(
            telemetry_inner,
            variable=self.progress_var,
            maximum=100.0,
            mode="determinate",
            cursor="hand2"
        )
        self.progressbar.pack(fill="x", pady=2)
        self.progressbar.bind("<Button-1>", lambda event: self._open_individual_progress_modal())

        lbl_active = tk.Label(
            telemetry_inner,
            textvariable=self.active_item_var,
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT_SECONDARY,
            bg=self.COLOR_CARD
        )
        lbl_active.pack(anchor="w", pady=(1, 0))

        # 3. Right Panel Tabbed Workspace (Pack LAST with expand=True so it fills remaining top space)
        self.notebook = ttk.Notebook(right_panel)
        self.notebook.pack(side="top", fill="both", expand=True, pady=(0, 2))

        # Tab 1: Video Queue & Batch Selector with Video Inspector Card
        self.tab_videos = tk.Frame(self.notebook, bg=self.COLOR_CARD)
        self.notebook.add(self.tab_videos, text="  Videos & Batch Queue (0)  ")

        # Tab 2: Direct CDN Stream Links Hub
        self.tab_cdn = tk.Frame(self.notebook, bg=self.COLOR_CARD)
        self.notebook.add(self.tab_cdn, text="  Direct CDN Stream Links (0)  ")

        self._build_tab_videos_ui()
        self._build_tab_cdn_ui()

    # --------------------------------------------------------------------------
    # TAB 1: VIDEOS & BATCH SELECTOR (With Deduplication, Filter, Quality & Size)
    # --------------------------------------------------------------------------

    def _build_tab_videos_ui(self):
        """Builds Video Queue tab with thumbnails, individual quality dropdown, deduplication, and filter."""
        # Top Action & Batch Toolbar
        toolbar_top = tk.Frame(self.tab_videos, bg=self.COLOR_CARD)
        toolbar_top.pack(fill="x", padx=8, pady=(6, 3))

        self.btn_download_batch = tk.Button(
            toolbar_top,
            text="START BATCH DOWNLOAD (0 SELECTED)",
            font=("Segoe UI", 9, "bold"),
            fg="#FFFFFF",
            bg=self.COLOR_SUCCESS,
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_SUCCESS_HOVER,
            cursor="hand2",
            bd=0,
            padx=12,
            pady=4,
            command=self._download_selected_videos
        )
        self.btn_download_batch.pack(side="left", padx=(0, 6))

        btn_sel_all = tk.Button(
            toolbar_top,
            text="Select All",
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT,
            bg="#1E293B",
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_PRIMARY,
            cursor="hand2",
            bd=0,
            padx=8,
            pady=4,
            command=self._select_all_items
        )
        btn_sel_all.pack(side="left", padx=2)

        btn_desel_all = tk.Button(
            toolbar_top,
            text="Deselect All",
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT,
            bg="#1E293B",
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_PRIMARY,
            cursor="hand2",
            bd=0,
            padx=8,
            pady=4,
            command=self._deselect_all_items
        )
        btn_desel_all.pack(side="left", padx=2)

        btn_invert = tk.Button(
            toolbar_top,
            text="Invert",
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT,
            bg="#1E293B",
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_PRIMARY,
            cursor="hand2",
            bd=0,
            padx=8,
            pady=4,
            command=self._invert_selection
        )
        btn_invert.pack(side="left", padx=2)

        # Remove Duplicates Button
        btn_dedup = tk.Button(
            toolbar_top,
            text="Remove Duplicates",
            font=("Segoe UI", 8, "bold"),
            fg=self.COLOR_TEXT,
            bg="#D97706",
            activeforeground="#FFFFFF",
            activebackground="#B45309",
            cursor="hand2",
            bd=0,
            padx=8,
            pady=4,
            command=self._remove_duplicates
        )
        btn_dedup.pack(side="left", padx=(6, 2))

        self.lbl_selected_count = tk.Label(
            toolbar_top,
            text="Items: 0 | Selected: 0",
            font=("Segoe UI", 8, "bold"),
            fg="#38BDF8",
            bg=self.COLOR_CARD
        )
        self.lbl_selected_count.pack(side="right", padx=6)

        # Secondary Filter & Batch Quality Row
        sub_bar = tk.Frame(self.tab_videos, bg=self.COLOR_CARD)
        sub_bar.pack(fill="x", padx=8, pady=(0, 4))

        # Search / Filter Entry
        lbl_search = tk.Label(sub_bar, text="Search / Filter:", font=("Segoe UI", 8, "bold"), fg=self.COLOR_TEXT_SECONDARY, bg=self.COLOR_CARD)
        lbl_search.pack(side="left", padx=(0, 4))

        ent_filter = tk.Entry(
            sub_bar,
            textvariable=self.filter_var,
            font=("Segoe UI", 8),
            bg=self.COLOR_INPUT_BG,
            fg="#F8FAFC",
            insertbackground="#FFFFFF",
            relief="solid",
            bd=1,
            width=18
        )
        ent_filter.pack(side="left", ipady=2, padx=(0, 8))

        # Batch quality selector
        lbl_apply_q = tk.Label(
            sub_bar,
            text="Set Quality for Selected:",
            font=("Segoe UI", 8, "bold"),
            fg=self.COLOR_TEXT_SECONDARY,
            bg=self.COLOR_CARD
        )
        lbl_apply_q.pack(side="left", padx=(6, 4))

        self.cbo_batch_quality = ttk.Combobox(
            sub_bar,
            textvariable=self.batch_apply_quality_var,
            values=self.QUALITY_OPTIONS,
            state="readonly",
            width=15,
            cursor="hand2"
        )
        self.cbo_batch_quality.pack(side="left", padx=2)

        btn_apply_qual = tk.Button(
            sub_bar,
            text="Apply to Selected",
            font=("Segoe UI", 8, "bold"),
            fg=self.COLOR_TEXT,
            bg="#334155",
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_PRIMARY,
            cursor="hand2",
            bd=0,
            padx=8,
            pady=2,
            command=self._apply_quality_to_selected
        )
        btn_apply_qual.pack(side="left", padx=4)

        # Simultaneous Downloads Selector in Tab 1 Toolbar
        lbl_tab_conc = tk.Label(
            sub_bar,
            text="Download Together:",
            font=("Segoe UI", 8, "bold"),
            fg=self.COLOR_TEXT_SECONDARY,
            bg=self.COLOR_CARD
        )
        lbl_tab_conc.pack(side="left", padx=(10, 4))

        cbo_tab_conc = ttk.Combobox(
            sub_bar,
            textvariable=self.concurrent_downloads_var,
            values=self.CONCURRENT_OPTIONS,
            state="readonly",
            width=14,
            cursor="hand2"
        )
        cbo_tab_conc.pack(side="left", padx=2)

        btn_tab_player = tk.Button(
            sub_bar,
            text="Preview Selected",
            font=("Segoe UI", 8, "bold"),
            fg="#FFFFFF",
            bg=self.COLOR_PRIMARY,
            activeforeground="#FFFFFF",
            activebackground="#0369A1",
            cursor="hand2",
            bd=0,
            padx=8,
            pady=2,
            command=self._open_custom_media_player
        )
        btn_tab_player.pack(side="left", padx=(8, 2))

        # Aggregate summary label
        self.lbl_summary_stats = tk.Label(
            sub_bar,
            text="Est. Total: ~0 MB (00:00)",
            font=("Segoe UI", 8),
            fg="#94A3B8",
            bg=self.COLOR_CARD
        )
        self.lbl_summary_stats.pack(side="right", padx=6)

        # Center Splitter: Left Table (Flexible), Right Video Inspector Card (280px)
        center_split = tk.Frame(self.tab_videos, bg=self.COLOR_CARD)
        center_split.pack(fill="both", expand=True, padx=8, pady=(0, 4))

        table_container = tk.Frame(center_split, bg=self.COLOR_CARD)
        table_container.pack(side="left", fill="both", expand=True)

        inspector_card = tk.LabelFrame(
            center_split,
            text=" Selected Video Inspector ",
            font=("Segoe UI", 9, "bold"),
            fg=self.COLOR_TEXT,
            bg=self.COLOR_CARD,
            bd=1,
            relief="solid",
            width=280
        )
        inspector_card.pack(side="right", fill="y", padx=(6, 0))
        inspector_card.pack_propagate(False)

        # Inside Inspector Card: Larger Thumbnail, Title, Duration, Size, Quality Dropdown, Open in Browser
        self.lbl_inspector_thumb = tk.Label(inspector_card, bg="#0F172A", relief="solid", bd=1)
        self.lbl_inspector_thumb.pack(fill="x", padx=8, pady=(6, 3))
        if self.placeholder_img:
            self.lbl_inspector_thumb.configure(image=self.placeholder_img)

        lbl_insp_title = tk.Label(
            inspector_card,
            textvariable=self.active_title_var,
            font=("Segoe UI", 8, "bold"),
            fg=self.COLOR_TEXT,
            bg=self.COLOR_CARD,
            wraplength=250,
            justify="left"
        )
        lbl_insp_title.pack(anchor="w", padx=8, pady=(2, 2))

        lbl_insp_info = tk.Label(
            inspector_card,
            textvariable=self.active_info_var,
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT_SECONDARY,
            bg=self.COLOR_CARD,
            wraplength=250,
            justify="left"
        )
        lbl_insp_info.pack(anchor="w", padx=8, pady=(0, 4))

        lbl_indiv_q = tk.Label(
            inspector_card,
            text="Individual Video Quality:",
            font=("Segoe UI", 8, "bold"),
            fg="#38BDF8",
            bg=self.COLOR_CARD
        )
        lbl_indiv_q.pack(anchor="w", padx=8, pady=(2, 2))

        self.cbo_indiv_quality = ttk.Combobox(
            inspector_card,
            textvariable=self.active_quality_var,
            values=self.QUALITY_OPTIONS,
            state="readonly",
            cursor="hand2"
        )
        self.cbo_indiv_quality.pack(fill="x", padx=8, pady=2)
        self.cbo_indiv_quality.bind("<<ComboboxSelected>>", self._on_individual_quality_changed)

        btn_toggle_active = tk.Button(
            inspector_card,
            text="Toggle Download Status",
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT,
            bg="#334155",
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_PRIMARY,
            cursor="hand2",
            bd=0,
            pady=4,
            command=self._toggle_active_video_status
        )
        btn_toggle_active.pack(fill="x", padx=8, pady=(6, 2))

        btn_open_browser = tk.Button(
            inspector_card,
            text="Open in Browser",
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT,
            bg="#1E293B",
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_CARD_HOVER,
            cursor="hand2",
            bd=0,
            pady=3,
            command=self._open_active_video_in_browser
        )
        btn_open_browser.pack(fill="x", padx=8, pady=(2, 0))

        btn_preview_player = tk.Button(
            inspector_card,
            text="Preview in Media Player",
            font=("Segoe UI", 8, "bold"),
            fg="#FFFFFF",
            bg=self.COLOR_PRIMARY,
            activeforeground="#FFFFFF",
            activebackground="#0369A1",
            cursor="hand2",
            bd=0,
            pady=4,
            command=self._open_custom_media_player
        )
        btn_preview_player.pack(fill="x", padx=8, pady=(4, 0))

        # Treeview Table for Videos (With highlighted row background tags!)
        tree_cols = ("selected", "title", "quality", "size", "duration", "uploader")
        self.tree = ttk.Treeview(table_container, columns=tree_cols, show="tree headings", selectmode="extended")

        self.tree.heading("#0", text="Thumb", anchor="center")
        self.tree.heading("selected", text="Download?", command=lambda: self._sort_tree("selected", False))
        self.tree.heading("title", text="Video Title", command=lambda: self._sort_tree("title", False))
        self.tree.heading("quality", text="Quality", command=lambda: self._sort_tree("quality", False))
        self.tree.heading("size", text="Est. Size", command=lambda: self._sort_tree("size", False))
        self.tree.heading("duration", text="Duration", command=lambda: self._sort_tree("duration", False))
        self.tree.heading("uploader", text="Channel", command=lambda: self._sort_tree("uploader", False))

        self.tree.column("#0", width=72, minwidth=65, stretch=False, anchor="center")
        self.tree.column("selected", width=75, minwidth=65, stretch=False, anchor="center")
        self.tree.column("title", width=250, minwidth=150, stretch=True, anchor="w")
        self.tree.column("quality", width=95, minwidth=80, stretch=False, anchor="center")
        self.tree.column("size", width=80, minwidth=70, stretch=False, anchor="center")
        self.tree.column("duration", width=70, minwidth=60, stretch=False, anchor="center")
        self.tree.column("uploader", width=120, minwidth=90, stretch=False, anchor="w")

        # Treeview Tags for selected vs unselected background highlighting
        self.tree.tag_configure("row_checked", background=self.COLOR_ROW_CHECKED, foreground="#FFFFFF")
        self.tree.tag_configure("row_unchecked", background=self.COLOR_ROW_UNCHECKED, foreground=self.COLOR_TEXT_MUTED)

        tree_scroll_y = ttk.Scrollbar(table_container, orient="vertical", command=self.tree.yview)
        self.tree.configure(yscrollcommand=tree_scroll_y.set)

        self.tree.pack(side="left", fill="both", expand=True)
        tree_scroll_y.pack(side="right", fill="y")

        self.tree.bind("<<TreeviewSelect>>", self._on_tree_row_selected)
        self.tree.bind("<Double-1>", self._toggle_tree_item_selection)
        self.tree.bind("<space>", self._toggle_tree_item_selection)

        # Context Menu for right-click on video rows
        self.tree_context_menu = tk.Menu(self.root, tearoff=0, bg="#1E293B", fg="#F8FAFC", activebackground=self.COLOR_PRIMARY, activeforeground="#FFFFFF")
        self.tree_context_menu.add_command(label="Preview in Media Player", command=self._open_custom_media_player)
        self.tree_context_menu.add_command(label="Open in Web Browser", command=self._open_active_video_in_browser)
        self.tree_context_menu.add_separator()
        self.tree_context_menu.add_command(label="Toggle Selection [YES/NO]", command=self._toggle_tree_item_selection)
        self.tree.bind("<Button-3>", self._show_tree_context_menu)

    # --------------------------------------------------------------------------
    # TAB 2: DIRECT CDN STREAM LINKS HUB (With Quality Selector & Clipboard Copy)
    # --------------------------------------------------------------------------

    def _build_tab_cdn_ui(self):
        """Builds Direct CDN Links tab with quality selector, multi-select, and clipboard copy."""
        cdn_toolbar = tk.Frame(self.tab_cdn, bg=self.COLOR_CARD)
        cdn_toolbar.pack(fill="x", padx=8, pady=5)

        btn_copy_selected = tk.Button(
            cdn_toolbar,
            text="COPY SELECTED TO CLIPBOARD",
            font=("Segoe UI", 9, "bold"),
            fg="#FFFFFF",
            bg=self.COLOR_PRIMARY,
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_PRIMARY_HOVER,
            cursor="hand2",
            bd=0,
            padx=12,
            pady=4,
            command=self._copy_selected_cdn_links
        )
        btn_copy_selected.pack(side="left", padx=(0, 6))

        btn_copy_all = tk.Button(
            cdn_toolbar,
            text="Copy All Links",
            font=("Segoe UI", 8, "bold"),
            fg=self.COLOR_TEXT,
            bg="#334155",
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_PRIMARY,
            cursor="hand2",
            bd=0,
            padx=8,
            pady=4,
            command=self._copy_all_cdn_links
        )
        btn_copy_all.pack(side="left", padx=2)

        btn_cdn_sel_all = tk.Button(
            cdn_toolbar,
            text="Select All",
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT,
            bg="#1E293B",
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_PRIMARY,
            cursor="hand2",
            bd=0,
            padx=8,
            pady=4,
            command=self._select_all_cdn_links
        )
        btn_cdn_sel_all.pack(side="left", padx=2)

        btn_cdn_desel_all = tk.Button(
            cdn_toolbar,
            text="Deselect All",
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT,
            bg="#1E293B",
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_PRIMARY,
            cursor="hand2",
            bd=0,
            padx=8,
            pady=4,
            command=self._deselect_all_cdn_links
        )
        btn_cdn_desel_all.pack(side="left", padx=2)

        self.lbl_cdn_count = tk.Label(
            cdn_toolbar,
            text="Links: 0 | Selected: 0",
            font=("Segoe UI", 8, "bold"),
            fg="#38BDF8",
            bg=self.COLOR_CARD
        )
        self.lbl_cdn_count.pack(side="right", padx=6)

        # Quality Selection Row for CDN Extractor
        cdn_qual_row = tk.Frame(self.tab_cdn, bg=self.COLOR_CARD)
        cdn_qual_row.pack(fill="x", padx=8, pady=(0, 4))

        lbl_cdn_q = tk.Label(
            cdn_qual_row,
            text="Stream Resolution Quality:",
            font=("Segoe UI", 8, "bold"),
            fg=self.COLOR_TEXT_SECONDARY,
            bg=self.COLOR_CARD
        )
        lbl_cdn_q.pack(side="left", padx=(0, 6))

        cbo_cdn_q = ttk.Combobox(
            cdn_qual_row,
            textvariable=self.cdn_quality_var,
            values=self.CDN_QUALITY_OPTIONS,
            state="readonly",
            width=18,
            cursor="hand2"
        )
        cbo_cdn_q.pack(side="left", padx=2)

        btn_reextract = tk.Button(
            cdn_qual_row,
            text="Extract / Refresh CDN Links",
            font=("Segoe UI", 8, "bold"),
            fg="#FFFFFF",
            bg="#0284C7",
            activeforeground="#FFFFFF",
            activebackground="#0369A1",
            cursor="hand2",
            bd=0,
            padx=10,
            pady=3,
            command=self._extract_cdn_with_chosen_quality
        )
        btn_reextract.pack(side="left", padx=6)

        # CDN Treeview Table (With highlighted row background tags!)
        cdn_tree_frame = tk.Frame(self.tab_cdn, bg=self.COLOR_CARD)
        cdn_tree_frame.pack(fill="both", expand=True, padx=8, pady=(0, 4))

        cdn_cols = ("idx", "selected", "title", "resolution", "url")
        self.tree_cdn = ttk.Treeview(cdn_tree_frame, columns=cdn_cols, show="headings", selectmode="extended")

        self.tree_cdn.heading("idx", text="#", command=lambda: self._sort_cdn_tree("idx", False))
        self.tree_cdn.heading("selected", text="Copy?", command=lambda: self._sort_cdn_tree("selected", False))
        self.tree_cdn.heading("title", text="Video Title", command=lambda: self._sort_cdn_tree("title", False))
        self.tree_cdn.heading("resolution", text="Quality / Codec", command=lambda: self._sort_cdn_tree("resolution", False))
        self.tree_cdn.heading("url", text="Direct GoogleVideo Stream URL (Double-Click / Space to Toggle)", command=lambda: self._sort_cdn_tree("url", False))

        self.tree_cdn.column("idx", width=42, anchor="center")
        self.tree_cdn.column("selected", width=70, anchor="center")
        self.tree_cdn.column("title", width=250, anchor="w")
        self.tree_cdn.column("resolution", width=95, anchor="center")
        self.tree_cdn.column("url", width=340, anchor="w")

        # Treeview Tags for selected vs unselected background highlighting
        self.tree_cdn.tag_configure("row_checked", background=self.COLOR_ROW_CHECKED, foreground="#FFFFFF")
        self.tree_cdn.tag_configure("row_unchecked", background=self.COLOR_ROW_UNCHECKED, foreground=self.COLOR_TEXT_MUTED)

        cdn_scroll_y = ttk.Scrollbar(cdn_tree_frame, orient="vertical", command=self.tree_cdn.yview)
        self.tree_cdn.configure(yscrollcommand=cdn_scroll_y.set)

        self.tree_cdn.pack(side="left", fill="both", expand=True)
        cdn_scroll_y.pack(side="right", fill="y")

        self.tree_cdn.bind("<Double-1>", self._toggle_cdn_item_selection)
        self.tree_cdn.bind("<space>", self._toggle_cdn_item_selection)

        # Raw URL List Multiline Preview Area
        preview_sec = tk.Frame(self.tab_cdn, bg=self.COLOR_CARD)
        preview_sec.pack(fill="x", padx=8, pady=(2, 4))

        prev_header = tk.Frame(preview_sec, bg=self.COLOR_CARD)
        prev_header.pack(fill="x", pady=(0, 2))

        lbl_prev = tk.Label(
            prev_header,
            text="Raw Stream URLs Preview (Ready for IDM / aria2):",
            font=("Segoe UI", 8, "bold"),
            fg=self.COLOR_TEXT_SECONDARY,
            bg=self.COLOR_CARD
        )
        lbl_prev.pack(side="left")

        btn_copy_raw = tk.Button(
            prev_header,
            text="Copy Raw Box Text",
            font=("Segoe UI", 7, "bold"),
            fg=self.COLOR_TEXT,
            bg="#1E293B",
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_PRIMARY,
            cursor="hand2",
            bd=0,
            padx=6,
            command=self._copy_raw_cdn_box
        )
        btn_copy_raw.pack(side="right")

        self.txt_cdn_preview = tk.Text(
            preview_sec,
            height=3,
            bg=self.COLOR_INPUT_BG,
            fg="#38BDF8",
            font=("Consolas", 8),
            relief="solid",
            bd=1,
            wrap="none"
        )
        prev_scroll_x = ttk.Scrollbar(preview_sec, orient="horizontal", command=self.txt_cdn_preview.xview)
        self.txt_cdn_preview.configure(xscrollcommand=prev_scroll_x.set)
        self.txt_cdn_preview.pack(fill="x")
        prev_scroll_x.pack(fill="x")

    # --------------------------------------------------------------------------
    # MODE SWITCHING & DYNAMIC CONTROLS
    # --------------------------------------------------------------------------

    def _switch_mode(self, mode_key: str):
        """Switches the active operation mode and updates visual styling."""
        if self.is_running:
            messagebox.showwarning("Execution Busy", "Please wait for current operation to finish or cancel before switching modes.")
            return

        self.current_mode = mode_key
        for m_key, btn in self.mode_buttons.items():
            if m_key == mode_key:
                btn.configure(bg=self.COLOR_PRIMARY, fg="#FFFFFF")
            else:
                btn.configure(bg=self.COLOR_CARD, fg=self.COLOR_TEXT_SECONDARY)

        self._apply_mode_ui_state()
        self._append_log(f"[MODE] Switched active operation to: {self.MODE_TITLES.get(mode_key, mode_key)}")

        if mode_key == "direct_links":
            self.notebook.select(self.tab_cdn)
        else:
            self.notebook.select(self.tab_videos)

        # If entries are loaded, adapt quality & sizes to the newly switched mode
        if self.inspected_entries:
            if mode_key == "audio_extractor":
                codec = self.audio_codec_var.get().upper()
                for e in self.inspected_entries:
                    e["quality"] = codec
                    raw_b = estimate_size_bytes(e.get("duration_sec", 0), codec)
                    e["bytes"] = raw_b
                    e["size"] = format_bytes(raw_b)
                    iid = str(e["idx"])
                    if self.tree.exists(iid):
                        vals = list(self.tree.item(iid, "values"))
                        vals[2] = codec
                        vals[3] = e["size"]
                        self.tree.item(iid, values=vals)
                self.btn_download_batch.configure(text=f"EXTRACT SELECTED AUDIO ({len([e for e in self.inspected_entries if e.get('selected')])} SELECTED)")
                self._update_aggregate_summary()
            elif mode_key in ("playlist", "single_video", "picker"):
                res = self.res_var.get()
                for e in self.inspected_entries:
                    e["quality"] = f"{res}p" if res.isdigit() else res
                    raw_b = estimate_size_bytes(e.get("duration_sec", 0), res)
                    e["bytes"] = raw_b
                    e["size"] = format_bytes(raw_b)
                    iid = str(e["idx"])
                    if self.tree.exists(iid):
                        vals = list(self.tree.item(iid, "values"))
                        vals[2] = e["quality"]
                        vals[3] = e["size"]
                        self.tree.item(iid, values=vals)
                self.btn_download_batch.configure(text=f"START BATCH DOWNLOAD ({len([e for e in self.inspected_entries if e.get('selected')])} SELECTED)")
                self._update_aggregate_summary()

    def _apply_mode_ui_state(self):
        """Enables/disables settings relevant to the selected mode."""
        mode = self.current_mode

        for m_key, btn in self.mode_buttons.items():
            if m_key == mode:
                btn.configure(bg=self.COLOR_PRIMARY, fg="#FFFFFF")
            else:
                btn.configure(bg=self.COLOR_CARD, fg=self.COLOR_TEXT_SECONDARY)

        if mode == "playlist":
            self.video_opts_frame.pack(fill="x", padx=8, pady=3)
            self.audio_opts_frame.pack_forget()
            self.range_opts_frame.pack(fill="x", padx=8, pady=3)
            self.params_sec.configure(text=" Default Quality & Range Presets ")
            self.btn_inspect_playlist.configure(text="FETCH & LOAD PLAYLIST VIDEOS")
            if self.inspected_entries:
                self.btn_execute.configure(text="DOWNLOAD PLAYLIST (F5)")
                self.btn_download_batch.configure(text=f"START BATCH DOWNLOAD ({len([e for e in self.inspected_entries if e.get('selected')])} SELECTED)")
            else:
                self.btn_execute.configure(text="LOAD PLAYLIST & PREVIEW (F5)")

        elif mode == "single_video":
            self.video_opts_frame.pack(fill="x", padx=8, pady=3)
            self.audio_opts_frame.pack_forget()
            self.range_opts_frame.pack_forget()
            self.params_sec.configure(text=" Single Video Stream Presets ")
            self.btn_inspect_playlist.configure(text="INSPECT VIDEO DETAILS")
            self.btn_execute.configure(text="DOWNLOAD SINGLE VIDEO (F5)")

        elif mode == "audio_extractor":
            self.video_opts_frame.pack_forget()
            self.audio_opts_frame.pack(fill="x", padx=8, pady=3)
            self.range_opts_frame.pack(fill="x", padx=8, pady=3)
            self.params_sec.configure(text=" Audio Extraction & Bitrate Presets ")
            self.btn_inspect_playlist.configure(text="FETCH & LOAD PLAYLIST FOR AUDIO")
            if self.inspected_entries:
                self.btn_execute.configure(text="EXTRACT BATCH AUDIO (F5)")
                self.btn_download_batch.configure(text=f"EXTRACT SELECTED AUDIO ({len([e for e in self.inspected_entries if e.get('selected')])} SELECTED)")
            else:
                self.btn_execute.configure(text="LOAD PLAYLIST FOR AUDIO (F5)")

        elif mode == "picker":
            self.video_opts_frame.pack(fill="x", padx=8, pady=3)
            self.audio_opts_frame.pack_forget()
            self.range_opts_frame.pack_forget()
            self.params_sec.configure(text=" Video Quality for Selected Items ")
            self.btn_inspect_playlist.configure(text="FETCH & LOAD VIDEOS")
            if self.inspected_entries:
                self.btn_execute.configure(text="START BATCH DOWNLOAD (F5)")
                self.btn_download_batch.configure(text=f"START BATCH DOWNLOAD ({len([e for e in self.inspected_entries if e.get('selected')])} SELECTED)")
            else:
                self.btn_execute.configure(text="LOAD VIDEOS & PREVIEW (F5)")

        elif mode == "direct_links":
            self.video_opts_frame.pack(fill="x", padx=8, pady=3)
            self.audio_opts_frame.pack_forget()
            self.range_opts_frame.pack_forget()
            self.params_sec.configure(text=" Direct Link Stream Resolution ")
            self.btn_execute.configure(text="EXTRACT DIRECT CDN LINKS (F5)")

    # --------------------------------------------------------------------------
    # THREAD-SAFE QUEUE POLLING (Python 3.13 Compatible)
    # --------------------------------------------------------------------------

    def _poll_queue(self):
        """Polls worker thread messages from self.ui_queue on the main Tkinter thread."""
        try:
            while not self.ui_queue.empty():
                msg_type, data = self.ui_queue.get_nowait()
                if msg_type == "log":
                    self._append_log(str(data))
                elif msg_type == "status":
                    self.status_text_var.set(str(data))
                elif msg_type == "init_tasks":
                    self.download_tasks_state = {it["idx"]: it for it in data}
                    self._update_progress_modal_if_open()
                elif msg_type == "item_progress":
                    idx = data.get("idx")
                    if idx in self.download_tasks_state:
                        self.download_tasks_state[idx].update(data)
                    else:
                        self.download_tasks_state[idx] = data
                    self._update_progress_modal_if_open()
                elif msg_type == "progress":
                    pct = data.get("percent", 0.0)
                    self.progress_var.set(pct)
                    speed = data.get("speed", 0.0)
                    eta = data.get("eta", 0)
                    dl_bytes = data.get("downloaded_bytes", 0)
                    tot_bytes = data.get("total_bytes", 0)
                    c_items = data.get("completed_items", 0)
                    t_items = data.get("total_items", 0)
                    filename = data.get("filename", "")

                    speed_str = f"{format_bytes(speed)}/s" if speed > 0 else "0.0 KB/s"
                    eta_str = format_seconds(eta) if eta > 0 else "--:--"
                    size_str = f"{format_bytes(dl_bytes)} / {format_bytes(tot_bytes)}" if tot_bytes > 0 else format_bytes(dl_bytes)

                    telemetry_parts = [f"Progress: {pct:.1f}%"]
                    if t_items > 0:
                        telemetry_parts.append(f"Items: {c_items}/{t_items}")
                    if tot_bytes > 0 or dl_bytes > 0:
                        telemetry_parts.append(f"Size: {size_str}")
                    telemetry_parts.append(f"Speed: {speed_str}")
                    telemetry_parts.append(f"ETA: {eta_str}")

                    self.speed_text_var.set(" | ".join(telemetry_parts))
                    if filename:
                        self.active_item_var.set(f"Active: {filename}")
                elif msg_type == "metadata_ready":
                    self._populate_treeview(data)
                elif msg_type == "thumbnail_ready":
                    self._on_thumbnail_loaded(data)
                elif msg_type == "cdn_link_found":
                    self._append_cdn_tree_row(data)
                elif msg_type == "cdn_links_ready":
                    self._on_cdn_links_ready(data)
                elif msg_type == "complete":
                    self._on_operation_finished(data)
        except Exception:
            pass
        finally:
            self.root.after(30, self._poll_queue)

    def _append_log(self, text: str):
        """Appends log entry into the live terminal console."""
        self.txt_log.configure(state="normal")
        self.txt_log.insert("end", f"{text}\n")
        if self.auto_scroll_var.get():
            self.txt_log.see("end")
        self.txt_log.configure(state="disabled")

    # --------------------------------------------------------------------------
    # PLAYLIST METADATA INSPECTION, DEDUPLICATION & DYNAMIC FILTER
    # --------------------------------------------------------------------------

    def _fetch_metadata_inspect(self):
        """Fetches metadata non-blockingly and populates the table with thumbnails."""
        raw_url = self.url_var.get().strip()
        parsed_urls = parse_target_urls(raw_url)
        if not parsed_urls:
            messagebox.showwarning("Missing URL", "Please enter a valid YouTube video or playlist URL.")
            self.ent_url.focus_set()
            return

        if self.is_running:
            messagebox.showwarning("Execution Busy", "An operation is currently active.")
            return

        self._set_running_state(True)
        self.notebook.select(self.tab_videos)
        self._append_log(f"[INFO] Inspecting playlist targets ({len(parsed_urls)} URL/playlist)...")
        self.cancel_event.clear()

        def worker():
            try:
                entries = YouTubeDownloaderEngine.extract_playlist_metadata(parsed_urls, self.ui_queue, self.cancel_event)
                if entries:
                    self.ui_queue.put(("metadata_ready", entries))
                    self.ui_queue.put(("complete", {"success": True, "action": "inspect"}))

                    # Background thumbnail worker
                    for idx, entry in enumerate(entries, 1):
                        if self.cancel_event.is_set():
                            break
                        vid_id = entry.get('id')
                        thumb_url = entry.get('thumbnail') or (f"https://i.ytimg.com/vi/{vid_id}/default.jpg" if vid_id else None)
                        if thumb_url:
                            raw_bytes = YouTubeDownloaderEngine.fetch_thumbnail_bytes(thumb_url)
                            if raw_bytes:
                                self.ui_queue.put(("thumbnail_ready", {"idx": idx, "bytes": raw_bytes}))
                else:
                    self.ui_queue.put(("complete", {"success": False, "action": "inspect", "error": "No videos found in target URL."}))
            except Exception as e:
                self.ui_queue.put(("log", f"[ERROR] Metadata inspection exception: {str(e)}"))
                self.ui_queue.put(("complete", {"success": False, "action": "inspect", "error": str(e)}))

        self.worker_thread = threading.Thread(target=worker, daemon=True)
        self.worker_thread.start()

    def _populate_treeview(self, entries: List[Dict[str, Any]]):
        """Populates the videos treeview table with extracted items and sets row tags."""
        for item in self.tree.get_children():
            self.tree.delete(item)

        if self.current_mode == "audio_extractor":
            default_quality = self.audio_codec_var.get().upper()
        else:
            default_quality = f"{self.res_var.get()}p" if self.res_var.get().isdigit() else self.res_var.get()
        self.inspected_entries = []
        self.thumbnail_cache.clear()

        for idx, entry in enumerate(entries, 1):
            if not entry:
                continue
            title = entry.get('title', 'Unknown Title')
            dur_sec = entry.get('duration') or 0
            dur_str = format_seconds(dur_sec)
            uploader = entry.get('uploader') or entry.get('channel') or 'Unknown'
            vid_id = entry.get('id', 'N/A')
            url = entry.get('url') or f"https://www.youtube.com/watch?v={vid_id}"
            est_size = estimate_size(dur_sec, default_quality)

            raw_bytes = entry.get('filesize') or entry.get('filesize_approx') or estimate_size_bytes(dur_sec, default_quality)
            est_size = format_bytes(raw_bytes)

            entry_dict = {
                "idx": idx,
                "selected": True,
                "title": title,
                "duration": dur_str,
                "duration_sec": dur_sec,
                "size": est_size,
                "bytes": raw_bytes,
                "quality": default_quality,
                "uploader": uploader,
                "id": vid_id,
                "url": url,
                "thumbnail_url": entry.get('thumbnail')
            }
            self.inspected_entries.append(entry_dict)

            iid = str(idx)
            kwargs = {"image": self.placeholder_img} if self.placeholder_img else {}
            self.tree.insert(
                "",
                "end",
                iid=iid,
                text=f"{idx:02d}",
                values=("[YES]", title, default_quality, est_size, dur_str, uploader),
                tags=("row_checked",),
                **kwargs
            )

        count = len(self.inspected_entries)
        self.lbl_selected_count.configure(text=f"Items: {count} | Selected: {count}")
        if self.current_mode == "audio_extractor":
            self.btn_download_batch.configure(text=f"EXTRACT SELECTED AUDIO ({count} SELECTED)")
            self.btn_execute.configure(text="EXTRACT BATCH AUDIO (F5)")
        else:
            self.btn_download_batch.configure(text=f"START BATCH DOWNLOAD ({count} SELECTED)")
            self.btn_execute.configure(text="DOWNLOAD PLAYLIST (F5)")
        self.notebook.tab(self.tab_videos, text=f"  Videos & Batch Queue ({count})  ")
        self._update_aggregate_summary()

        if self.inspected_entries:
            self._select_inspector_video(0)

        self._append_log(f"[SUCCESS] Loaded {count} playlist item(s). Adjust individual qualities, select files, then click 'START BATCH DOWNLOAD'.")
        self.status_text_var.set(f"Loaded {count} videos. Select files and adjust qualities below.")

    def _remove_duplicates(self):
        """Scans the video queue, removes duplicate videos by ID or URL, and refreshes."""
        if not self.inspected_entries:
            messagebox.showinfo("No Items", "Queue is currently empty.")
            return

        seen_ids = set()
        seen_urls = set()
        unique_entries = []
        dups_removed = 0

        for entry in self.inspected_entries:
            vid_id = entry.get("id")
            url = entry.get("url")

            is_dup = False
            if vid_id and vid_id != "N/A":
                if vid_id in seen_ids:
                    is_dup = True
                else:
                    seen_ids.add(vid_id)

            if not is_dup and url:
                if url in seen_urls:
                    is_dup = True
                else:
                    seen_urls.add(url)

            if is_dup:
                dups_removed += 1
            else:
                unique_entries.append(entry)

        if dups_removed > 0:
            # Re-index unique entries
            for i, e in enumerate(unique_entries, 1):
                e["idx"] = i

            # Re-populate treeview
            for item in self.tree.get_children():
                self.tree.delete(item)

            for e in unique_entries:
                idx = e["idx"]
                iid = str(idx)
                tag = "row_checked" if e["selected"] else "row_unchecked"
                img = self.thumbnail_cache.get(idx, self.placeholder_img)
                kwargs = {"image": img} if img else {}
                self.tree.insert(
                    "",
                    "end",
                    iid=iid,
                    text=f"{idx:02d}",
                    values=("[YES]" if e["selected"] else "[NO]", e["title"], e["quality"], e["size"], e["duration"], e["uploader"]),
                    tags=(tag,),
                    **kwargs
                )

            self.inspected_entries = unique_entries
            self._update_selected_count_badge()
            self._update_aggregate_summary()
            self._append_log(f"[CLEANUP] Successfully removed {dups_removed} duplicate video(s). Remaining unique: {len(unique_entries)}")
            messagebox.showinfo("Deduplication Complete", f"Removed {dups_removed} duplicate video(s).\nRemaining unique items: {len(unique_entries)}")
        else:
            self._append_log("[CLEANUP] No duplicate videos found in queue.")
            messagebox.showinfo("No Duplicates", "No duplicate videos were found in the queue.")

    def _on_filter_changed(self):
        """Filters visible rows in the videos treeview based on keyword search."""
        query = self.filter_var.get().strip().lower()

        # Detach all and re-insert matching
        for item in self.tree.get_children():
            self.tree.detach(item)

        for entry in self.inspected_entries:
            idx = entry["idx"]
            iid = str(idx)
            title = entry["title"].lower()
            uploader = entry["uploader"].lower()
            if not query or query in title or query in uploader:
                self.tree.move(iid, '', 'end')

    def _on_thumbnail_loaded(self, data: Dict[str, Any]):
        """Callback when background thumbnail bytes arrive."""
        idx = data["idx"]
        raw = data["bytes"]
        if not PILLOW_AVAILABLE:
            return
        try:
            pil_img = Image.open(io.BytesIO(raw)).resize((64, 36), Image.Resampling.LANCZOS)
            tk_img = ImageTk.PhotoImage(pil_img)
            self.thumbnail_cache[idx] = tk_img
            iid = str(idx)
            if self.tree.exists(iid):
                self.tree.item(iid, image=tk_img)

            if self.active_entry_idx is not None and (self.active_entry_idx + 1) == idx:
                large_pil = Image.open(io.BytesIO(raw)).resize((180, 101), Image.Resampling.LANCZOS)
                self.active_large_thumb = ImageTk.PhotoImage(large_pil)
                self.lbl_inspector_thumb.configure(image=self.active_large_thumb)
        except Exception:
            pass

    def _on_tree_row_selected(self, event=None):
        """Called when a user clicks/selects a row in the videos treeview."""
        selected_ids = self.tree.selection()
        if not selected_ids:
            return
        first_iid = selected_ids[0]
        try:
            idx = int(first_iid) - 1
            if 0 <= idx < len(self.inspected_entries):
                self._select_inspector_video(idx)
        except ValueError:
            pass

    def _select_inspector_video(self, idx: int):
        """Populates the Selected Video Inspector card for chosen video index."""
        self.active_entry_idx = idx
        entry = self.inspected_entries[idx]

        self.active_title_var.set(f"#{idx+1:02d}: {entry['title']}")
        info_text = f"Duration: {entry['duration']} | Est. Size: {entry['size']}\nChannel: {entry['uploader']}"
        self.active_info_var.set(info_text)

        curr_q = entry.get("quality", "1080p")
        for opt in self.QUALITY_OPTIONS:
            if curr_q.lower() in opt.lower():
                self.active_quality_var.set(opt)
                break
        else:
            self.active_quality_var.set(f"{curr_q}")

        if idx + 1 in self.thumbnail_cache:
            self.lbl_inspector_thumb.configure(image=self.thumbnail_cache[idx + 1])
        elif self.placeholder_img:
            self.lbl_inspector_thumb.configure(image=self.placeholder_img)

    def _open_active_video_in_browser(self):
        """Opens the currently inspected video URL in the user's default browser."""
        if self.active_entry_idx is None or self.active_entry_idx >= len(self.inspected_entries):
            return
        url = self.inspected_entries[self.active_entry_idx].get("url")
        if url:
            webbrowser.open(url)
            self._append_log(f"[BROWSER] Opened video in browser: {url}")

    def _on_individual_quality_changed(self, event=None):
        """Called when the user modifies quality combobox in the Inspector card."""
        if self.active_entry_idx is None or self.active_entry_idx >= len(self.inspected_entries):
            return

        new_opt = self.active_quality_var.get()
        short_q = new_opt.split(" ")[0].replace("(", "").replace(")", "")
        entry = self.inspected_entries[self.active_entry_idx]
        entry["quality"] = short_q
        raw_b = estimate_size_bytes(entry["duration_sec"], short_q)
        entry["bytes"] = raw_b
        entry["size"] = format_bytes(raw_b)

        iid = str(self.active_entry_idx + 1)
        if self.tree.exists(iid):
            vals = list(self.tree.item(iid, "values"))
            vals[2] = short_q
            vals[3] = entry["size"]
            self.tree.item(iid, values=vals)

        info_text = f"Duration: {entry['duration']} | Est. Size: {entry['size']}\nChannel: {entry['uploader']}"
        self.active_info_var.set(info_text)
        self._update_aggregate_summary()
        self._append_log(f"[CONFIG] Updated Video #{self.active_entry_idx+1} quality to: {short_q} (Est. Size: {entry['size']})")

    def _apply_quality_to_selected(self):
        """Applies chosen quality from the batch toolbar to all highlighted treeview rows."""
        selected_iids = self.tree.selection()
        if not selected_iids:
            messagebox.showinfo("No Selection", "Please click/highlight one or more rows in the table to apply quality.")
            return

        new_opt = self.batch_apply_quality_var.get()
        short_q = new_opt.split(" ")[0].replace("(", "").replace(")", "")

        updated_count = 0
        for iid in selected_iids:
            try:
                idx = int(iid) - 1
                if 0 <= idx < len(self.inspected_entries):
                    entry = self.inspected_entries[idx]
                    entry["quality"] = short_q
                    entry["size"] = estimate_size(entry["duration_sec"], short_q)

                    vals = list(self.tree.item(iid, "values"))
                    vals[2] = short_q
                    vals[3] = entry["size"]
                    self.tree.item(iid, values=vals)
                    updated_count += 1
            except ValueError:
                pass

        self._update_aggregate_summary()
        self._append_log(f"[CONFIG] Applied quality '{short_q}' to {updated_count} selected video(s).")
        if self.active_entry_idx is not None:
            self._select_inspector_video(self.active_entry_idx)

    def _toggle_active_video_status(self):
        """Toggles the download status of currently inspected video."""
        if self.active_entry_idx is None or self.active_entry_idx >= len(self.inspected_entries):
            return
        iid = str(self.active_entry_idx + 1)
        entry = self.inspected_entries[self.active_entry_idx]
        entry["selected"] = not entry["selected"]

        if self.tree.exists(iid):
            vals = list(self.tree.item(iid, "values"))
            vals[0] = "[YES]" if entry["selected"] else "[NO]"
            tag = "row_checked" if entry["selected"] else "row_unchecked"
            self.tree.item(iid, values=vals, tags=(tag,))

        self._update_selected_count_badge()
        self._update_aggregate_summary()

    def _toggle_tree_item_selection(self, event=None):
        """Toggles the selected status of chosen treeview row(s)."""
        selected_ids = self.tree.selection()
        if not selected_ids:
            return

        for item_id in selected_ids:
            try:
                idx = int(item_id) - 1
                if 0 <= idx < len(self.inspected_entries):
                    curr = self.inspected_entries[idx]["selected"]
                    new_state = not curr
                    self.inspected_entries[idx]["selected"] = new_state
                    sel_str = "[YES]" if new_state else "[NO]"
                    tag = "row_checked" if new_state else "row_unchecked"

                    vals = list(self.tree.item(item_id, "values"))
                    vals[0] = sel_str
                    self.tree.item(item_id, values=vals, tags=(tag,))
            except ValueError:
                pass

        self._update_selected_count_badge()
        self._update_aggregate_summary()

    def _select_all_items(self):
        """Marks all items in the treeview as selected and applies checked highlight."""
        for i, item_dict in enumerate(self.inspected_entries, 1):
            item_dict["selected"] = True
            iid = str(i)
            if self.tree.exists(iid):
                vals = list(self.tree.item(iid, "values"))
                vals[0] = "[YES]"
                self.tree.item(iid, values=vals, tags=("row_checked",))
        self._update_selected_count_badge()
        self._update_aggregate_summary()

    def _deselect_all_items(self):
        """Marks all items in the treeview as unselected and removes checked highlight."""
        for i, item_dict in enumerate(self.inspected_entries, 1):
            item_dict["selected"] = False
            iid = str(i)
            if self.tree.exists(iid):
                vals = list(self.tree.item(iid, "values"))
                vals[0] = "[NO]"
                self.tree.item(iid, values=vals, tags=("row_unchecked",))
        self._update_selected_count_badge()
        self._update_aggregate_summary()

    def _invert_selection(self):
        """Inverts the selection status of every item."""
        for i, item_dict in enumerate(self.inspected_entries, 1):
            item_dict["selected"] = not item_dict["selected"]
            iid = str(i)
            if self.tree.exists(iid):
                vals = list(self.tree.item(iid, "values"))
                vals[0] = "[YES]" if item_dict["selected"] else "[NO]"
                tag = "row_checked" if item_dict["selected"] else "row_unchecked"
                self.tree.item(iid, values=vals, tags=(tag,))
        self._update_selected_count_badge()
        self._update_aggregate_summary()

    def _update_selected_count_badge(self):
        """Updates the count badge in the treeview toolbar."""
        total = len(self.inspected_entries)
        sel = sum(1 for e in self.inspected_entries if e["selected"])
        self.lbl_selected_count.configure(text=f"Items: {total} | Selected: {sel}")
        self.btn_download_batch.configure(text=f"START BATCH DOWNLOAD ({sel} SELECTED)")

    def _update_aggregate_summary(self):
        """Calculates total duration and estimated size of all selected videos strictly in Bytes."""
        total_sec = 0
        total_bytes = 0

        for e in self.inspected_entries:
            if e.get("selected"):
                total_sec += e.get("duration_sec", 0)
                b = e.get("bytes") or estimate_size_bytes(e.get("duration_sec", 0), e.get("quality", "1080"))
                total_bytes += b

        size_str = format_bytes(total_bytes)
        self.lbl_summary_stats.configure(text=f"Total: ~{size_str} ({format_seconds(total_sec)})")

    def _download_selected_videos(self):
        """Downloads all currently selected items from the table in batch."""
        selected_items = [e for e in self.inspected_entries if e["selected"]]
        if not selected_items:
            messagebox.showwarning("No Items Selected", "Please select at least one video (marked [YES]) to download.")
            return

        self._start_execution(custom_items=selected_items)

    def _show_multi_urls_dialog(self):
        """Opens a modal text area for pasting multiple URLs."""
        modal = tk.Toplevel(self.root)
        modal.title("Add Multiple Video / Playlist URLs")
        modal.geometry("560x420")
        modal.configure(bg=self.COLOR_BG)
        modal.transient(self.root)
        modal.grab_set()

        lbl_info = tk.Label(
            modal,
            text="Paste Multiple YouTube URLs (One per line or comma-separated):",
            font=("Segoe UI", 9, "bold"),
            fg=self.COLOR_TEXT,
            bg=self.COLOR_BG
        )
        lbl_info.pack(anchor="w", padx=16, pady=(12, 6))

        txt_frame = tk.Frame(modal, bg=self.COLOR_CARD)
        txt_frame.pack(fill="both", expand=True, padx=16, pady=4)

        txt_multi = tk.Text(
            txt_frame,
            bg=self.COLOR_INPUT_BG,
            fg="#F8FAFC",
            font=("Consolas", 9),
            relief="solid",
            bd=1,
            wrap="none"
        )
        s_y = ttk.Scrollbar(txt_frame, orient="vertical", command=txt_multi.yview)
        txt_multi.configure(yscrollcommand=s_y.set)
        txt_multi.pack(side="left", fill="both", expand=True)
        s_y.pack(side="right", fill="y")

        btn_row = tk.Frame(modal, bg=self.COLOR_BG)
        btn_row.pack(fill="x", padx=16, pady=12)

        def do_paste():
            try:
                txt_multi.insert("end", self.root.clipboard_get() + "\n")
            except Exception:
                pass

        def do_import():
            raw = txt_multi.get("1.0", "end")
            urls = parse_target_urls(raw)
            if not urls:
                messagebox.showwarning("Empty Input", "No valid URLs found in text area.", parent=modal)
                return
            self.url_var.set(urls[0] if len(urls) == 1 else ", ".join(urls))
            self._append_log(f"[INPUT] Imported {len(urls)} target URL(s) from batch dialog.")
            modal.destroy()
            self._fetch_metadata_inspect()

        tk.Button(
            btn_row,
            text="Paste from Clipboard",
            font=("Segoe UI", 9),
            fg=self.COLOR_TEXT,
            bg="#334155",
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_CARD_HOVER,
            cursor="hand2",
            bd=0,
            padx=12,
            pady=5,
            command=do_paste
        ).pack(side="left")

        tk.Button(
            btn_row,
            text="Add & Fetch Metadata",
            font=("Segoe UI", 9, "bold"),
            fg="#FFFFFF",
            bg=self.COLOR_PRIMARY,
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_PRIMARY_HOVER,
            cursor="hand2",
            bd=0,
            padx=14,
            pady=5,
            command=do_import
        ).pack(side="right")

    def _sort_tree(self, col: str, reverse: bool):
        """Sorts the videos treeview by column."""
        l = [(self.tree.set(k, col), k) for k in self.tree.get_children('')]
        try:
            l.sort(key=lambda t: int(t[0]), reverse=reverse)
        except ValueError:
            l.sort(reverse=reverse)

        for index, (val, k) in enumerate(l):
            self.tree.move(k, '', index)

        self.tree.heading(col, command=lambda: self._sort_tree(col, not reverse))

    # --------------------------------------------------------------------------
    # TAB 2: DIRECT CDN LINKS ACTIONS (Quality Selector & In-Dashboard Copy)
    # --------------------------------------------------------------------------

    def _extract_cdn_with_chosen_quality(self):
        """Triggers CDN stream extraction using the resolution quality chosen in Tab 2."""
        raw_url = self.url_var.get().strip()
        target_urls = parse_target_urls(raw_url)
        if not target_urls:
            messagebox.showwarning("Missing URL", "Please enter a valid YouTube video or playlist URL in the URL field.")
            self.ent_url.focus_set()
            return

        chosen_quality = self.cdn_quality_var.get()
        self.res_var.set(chosen_quality.split(" ")[0].replace("p", ""))
        self._append_log(f"[CONFIG] Starting CDN link extraction with selected quality: {chosen_quality}")

        self._set_running_state(True)
        self.cancel_event.clear()
        self.progress_var.set(0.0)

        # Clear existing CDN tree rows
        for item in self.tree_cdn.get_children():
            self.tree_cdn.delete(item)
        self.cdn_links.clear()

        def worker():
            try:
                YouTubeDownloaderEngine.extract_direct_cdn_links_worker(
                    target_urls,
                    chosen_quality,
                    self.ui_queue,
                    self.cancel_event
                )
                self.ui_queue.put(("complete", {"success": True, "action": "direct_links"}))
            except Exception as e:
                self.ui_queue.put(("log", f"[FATAL ERROR] {str(e)}"))
                self.ui_queue.put(("complete", {"success": False, "action": "error"}))

        self.worker_thread = threading.Thread(target=worker, daemon=True)
        self.worker_thread.start()

    def _append_cdn_tree_row(self, link_record: Dict[str, Any]):
        """Adds a newly extracted stream URL row to the CDN treeview with checked highlight."""
        idx = link_record["idx"]
        self.cdn_links.append(link_record)

        self.tree_cdn.insert(
            "",
            "end",
            iid=f"cdn_{idx}",
            values=(f"{idx:03d}", "[YES]", link_record["title"], link_record["resolution"], link_record["url"]),
            tags=("row_checked",)
        )
        self._update_cdn_preview_and_badge()

    def _on_cdn_links_ready(self, all_links: List[Dict[str, Any]]):
        """Called when direct CDN link extraction is complete."""
        self.cdn_links = all_links
        for item in self.tree_cdn.get_children():
            self.tree_cdn.delete(item)

        for link in all_links:
            idx = link["idx"]
            self.tree_cdn.insert(
                "",
                "end",
                iid=f"cdn_{idx}",
                values=(f"{idx:03d}", "[YES]", link["title"], link["resolution"], link["url"]),
                tags=("row_checked",)
            )

        self.notebook.select(self.tab_cdn)
        self.notebook.tab(self.tab_cdn, text=f"  Direct CDN Stream Links ({len(all_links)})  ")
        self._update_cdn_preview_and_badge()
        self._append_log(f"[INFO] Direct CDN stream links ready. Click 'COPY SELECTED TO CLIPBOARD' to copy.")

    def _toggle_cdn_item_selection(self, event=None):
        """Toggles copy selection status for chosen row(s) in the CDN table."""
        selected_ids = self.tree_cdn.selection()
        if not selected_ids:
            return

        for item_id in selected_ids:
            raw_idx = item_id.replace("cdn_", "")
            try:
                idx = int(raw_idx) - 1
                if 0 <= idx < len(self.cdn_links):
                    curr = self.cdn_links[idx]["selected"]
                    new_state = not curr
                    self.cdn_links[idx]["selected"] = new_state
                    sel_str = "[YES]" if new_state else "[NO]"
                    tag = "row_checked" if new_state else "row_unchecked"

                    vals = list(self.tree_cdn.item(item_id, "values"))
                    vals[1] = sel_str
                    self.tree_cdn.item(item_id, values=vals, tags=(tag,))
            except ValueError:
                pass

        self._update_cdn_preview_and_badge()

    def _select_all_cdn_links(self):
        """Selects all links in the CDN table."""
        for i, link in enumerate(self.cdn_links, 1):
            link["selected"] = True
            iid = f"cdn_{i}"
            if self.tree_cdn.exists(iid):
                vals = list(self.tree_cdn.item(iid, "values"))
                vals[1] = "[YES]"
                self.tree_cdn.item(iid, values=vals, tags=("row_checked",))
        self._update_cdn_preview_and_badge()

    def _deselect_all_cdn_links(self):
        """Deselects all links in the CDN table."""
        for i, link in enumerate(self.cdn_links, 1):
            link["selected"] = False
            iid = f"cdn_{i}"
            if self.tree_cdn.exists(iid):
                vals = list(self.tree_cdn.item(iid, "values"))
                vals[1] = "[NO]"
                self.tree_cdn.item(iid, values=vals, tags=("row_unchecked",))
        self._update_cdn_preview_and_badge()

    def _update_cdn_preview_and_badge(self):
        """Updates raw preview box and count badge."""
        total = len(self.cdn_links)
        selected_links = [l for l in self.cdn_links if l["selected"]]
        sel_count = len(selected_links)

        self.lbl_cdn_count.configure(text=f"Links: {total} | Selected: {sel_count}")

        self.txt_cdn_preview.delete("1.0", "end")
        for l in selected_links:
            self.txt_cdn_preview.insert("end", f"{l['url']}\n")

    def _copy_selected_cdn_links(self):
        """Copies all selected direct CDN stream URLs to the system clipboard."""
        selected_urls = [l["url"] for l in self.cdn_links if l["selected"]]
        if not selected_urls:
            messagebox.showwarning("No Links Selected", "Please select at least one link (marked [YES]) to copy.")
            return

        combined_text = "\n".join(selected_urls)
        self.root.clipboard_clear()
        self.root.clipboard_append(combined_text)

        count = len(selected_urls)
        self.status_text_var.set(f"Copied {count} direct stream link(s) to clipboard!")
        self._append_log(f"[COPIED] Successfully copied {count} direct stream URL(s) to clipboard.")
        messagebox.showinfo("Copied to Clipboard", f"Successfully copied {count} direct CDN stream link(s) to your clipboard.\nReady to paste into IDM or aria2!")

    def _copy_all_cdn_links(self):
        """Copies all extracted direct CDN stream URLs to the system clipboard."""
        if not self.cdn_links:
            messagebox.showwarning("No Links", "No direct CDN stream links have been extracted yet.")
            return

        all_urls = [l["url"] for l in self.cdn_links]
        combined_text = "\n".join(all_urls)
        self.root.clipboard_clear()
        self.root.clipboard_append(combined_text)

        count = len(all_urls)
        self.status_text_var.set(f"Copied all {count} direct stream link(s) to clipboard!")
        self._append_log(f"[COPIED] Successfully copied all {count} direct stream URL(s) to clipboard.")
        messagebox.showinfo("Copied to Clipboard", f"Successfully copied all {count} direct CDN stream link(s) to your clipboard.")

    def _copy_raw_cdn_box(self):
        """Copies whatever is in the raw CDN preview text box."""
        raw = self.txt_cdn_preview.get("1.0", "end").strip()
        if not raw:
            return
        self.root.clipboard_clear()
        self.root.clipboard_append(raw)
        self._append_log("[COPIED] Raw stream URLs copied to clipboard.")

    def _sort_cdn_tree(self, col: str, reverse: bool):
        """Sorts the CDN links treeview by column."""
        l = [(self.tree_cdn.set(k, col), k) for k in self.tree_cdn.get_children('')]
        try:
            l.sort(key=lambda t: int(t[0]), reverse=reverse)
        except ValueError:
            l.sort(reverse=reverse)

        for index, (val, k) in enumerate(l):
            self.tree_cdn.move(k, '', index)

        self.tree_cdn.heading(col, command=lambda: self._sort_cdn_tree(col, not reverse))

    # --------------------------------------------------------------------------
    # EXECUTION PIPELINE
    # --------------------------------------------------------------------------

    def _start_execution(self, custom_items: Optional[List[Dict[str, Any]]] = None):
        """Main execution dispatcher."""
        if self.is_running:
            return

        raw_url = self.url_var.get().strip()
        target_urls = parse_target_urls(raw_url)
        out_dir = self.out_dir_var.get().strip()
        mode = self.current_mode

        # If in Playlist, Audio Extractor, or Picker mode and table is not loaded yet, fetch and preview first!
        if mode in ("playlist", "audio_extractor", "picker") and not custom_items and not self.inspected_entries:
            self._fetch_metadata_inspect()
            return

        if not custom_items and not target_urls:
            messagebox.showwarning("Missing Input", "Please enter a valid YouTube URL or playlist.")
            self.ent_url.focus_set()
            return

        if not out_dir:
            messagebox.showwarning("Missing Output Directory", "Please select an output folder.")
            return

        # Prepare items to download
        if custom_items:
            items_to_download = custom_items
        elif self.inspected_entries:
            items_to_download = [e for e in self.inspected_entries if e["selected"]]
            if not items_to_download:
                messagebox.showwarning("No Items Selected", "Please select at least one video (marked [YES]) from the list.")
                return
        else:
            default_q = self.audio_codec_var.get() if mode == "audio_extractor" else self.res_var.get()
            items_to_download = [{
                "title": "Direct Audio Stream" if mode == "audio_extractor" else "Direct Video",
                "url": target_urls[0],
                "quality": default_q
            }]

        if mode == "audio_extractor":
            codec = self.audio_codec_var.get()
            for it in items_to_download:
                q_val = str(it.get("quality", "")).lower()
                if not q_val or q_val in ("best", "1080", "720", "480", "360", "2160", "1440") or q_val.endswith("p"):
                    it["quality"] = codec

        # Speed limit parsing
        limit_map = {
            "Unlimited": None,
            "10 MB/s": 10 * 1024 * 1024,
            "5 MB/s": 5 * 1024 * 1024,
            "2 MB/s": 2 * 1024 * 1024,
            "1 MB/s": 1 * 1024 * 1024
        }
        rate_limit = limit_map.get(self.speed_limit_var.get())

        # Concurrency parsing (How many videos to download together)
        raw_concurrency = self.concurrent_downloads_var.get()
        concurrency = 1
        for token in raw_concurrency.split():
            if token.isdigit():
                concurrency = max(1, int(token))
                break

        global_config = {
            "mode": mode,
            "out_dir": out_dir,
            "resolution": self.res_var.get(),
            "container": self.container_var.get(),
            "audio_codec": self.audio_codec_var.get(),
            "audio_quality": self.audio_quality_var.get(),
            "numbering_index": self.numbering_var.get(),
            "embed_thumbnail": self.thumbnail_var.get(),
            "embed_metadata": self.metadata_var.get(),
            "write_subtitles": self.subtitles_var.get(),
            "rate_limit": rate_limit,
            "concurrency": concurrency
        }

        self._set_running_state(True)
        self.cancel_event.clear()
        self.progress_var.set(0.0)

        self._append_log("=" * 65)
        self._append_log(f"Starting Operation: {self.MODE_TITLES.get(mode, mode)}")
        self._append_log(f"Items to Process: {len(items_to_download)} video(s)")
        self._append_log(f"Simultaneous Downloads: {concurrency} video(s) at once")
        self._append_log(f"Destination: {out_dir}")
        self._append_log("=" * 65)

        def worker():
            try:
                if mode == "direct_links":
                    YouTubeDownloaderEngine.extract_direct_cdn_links_worker(
                        target_urls,
                        self.cdn_quality_var.get(),
                        self.ui_queue,
                        self.cancel_event
                    )
                    self.ui_queue.put(("complete", {"success": True, "action": "direct_links"}))
                else:
                    success = YouTubeDownloaderEngine.execute_download(
                        items_to_download,
                        global_config,
                        self.ui_queue,
                        self.cancel_event
                    )
                    self.ui_queue.put(("complete", {"success": success, "action": "download", "out_dir": out_dir}))
            except Exception as e:
                self.ui_queue.put(("log", f"[FATAL ERROR] {str(e)}"))
                self.ui_queue.put(("complete", {"success": False, "action": "error"}))

        self.worker_thread = threading.Thread(target=worker, daemon=True)
        self.worker_thread.start()

    def _cancel_execution(self):
        """Signals cancellation to the running worker thread."""
        if not self.is_running:
            return
        self.cancel_event.set()
        self.status_text_var.set("Cancelling operation...")
        self._append_log("[ACTION] Cancel signal sent by user. Terminating process...")

    def _on_operation_finished(self, result: Dict[str, Any]):
        """Handles post-completion logic on the main thread."""
        self._set_running_state(False)
        success = result.get("success", False)
        action = result.get("action", "")

        if action == "inspect":
            if success:
                count = len(self.inspected_entries)
                self.status_text_var.set(f"Loaded {count} videos! Select items, adjust qualities, and click 'START BATCH DOWNLOAD'.")
                self.btn_execute.configure(text=f"START BATCH DOWNLOAD ({count} SELECTED)")
                self.active_item_var.set(f"Active: {count} video(s) ready in queue")
            else:
                err_msg = result.get("error", "Check URL connectivity or verify playlist visibility.")
                self.status_text_var.set(f"Metadata inspection failed: {err_msg}")
                self.active_item_var.set("Active: Inspection failed")
            return

        if action == "direct_links":
            if success:
                count = len(self.cdn_links)
                self.progress_var.set(100.0)
                self.status_text_var.set(f"Direct CDN links extracted! Ready to copy ({count} links).")
                self.active_item_var.set(f"Active: {count} direct stream link(s) ready")
            else:
                self.status_text_var.set("Direct CDN link extraction cancelled or failed.")
                self.active_item_var.set("Active: Extraction failed")
            return

        if success:
            self.progress_var.set(100.0)
            self.status_text_var.set("Operation completed successfully!")
            self._append_log("[COMPLETED] All downloads and post-processing steps finished.")

            if self.open_dest_var.get():
                out_dir = result.get("out_dir") or self.out_dir_var.get()
                if os.path.isdir(out_dir):
                    try:
                        os.startfile(out_dir)
                    except Exception:
                        pass
        else:
            self.status_text_var.set("Operation cancelled or failed.")

    def _set_running_state(self, is_running: bool):
        """Toggles controls between active and busy states."""
        self.is_running = is_running
        if is_running:
            self.btn_execute.configure(state="disabled", cursor="")
            self.btn_download_batch.configure(state="disabled", cursor="")
            self.btn_inspect_playlist.configure(state="disabled", cursor="")
            self.btn_cancel.configure(state="normal", cursor="hand2")
            for btn in self.mode_buttons.values():
                btn.configure(state="disabled", cursor="")
        else:
            self.btn_execute.configure(state="normal", cursor="hand2")
            self.btn_download_batch.configure(state="normal", cursor="hand2")
            self.btn_inspect_playlist.configure(state="normal", cursor="hand2")
            self.btn_cancel.configure(state="disabled", cursor="")
            for btn in self.mode_buttons.values():
                btn.configure(state="normal", cursor="hand2")

    # --------------------------------------------------------------------------
    # UTILITY ACTIONS & SHORTCUTS
    # --------------------------------------------------------------------------

    def _paste_url(self):
        """Pastes URL from clipboard into URL entry."""
        try:
            clip = self.root.clipboard_get().strip()
            if clip:
                self.url_var.set(clip)
                self._append_log(f"[INPUT] Pasted URL: {clip}")
        except Exception:
            pass

    def _browse_output_dir(self):
        """Browses and selects destination output folder."""
        curr = self.out_dir_var.get()
        chosen = filedialog.askdirectory(initialdir=curr if os.path.isdir(curr) else str(Path.home()))
        if chosen:
            self.out_dir_var.set(chosen)
            self._append_log(f"[CONFIG] Destination updated to: {chosen}")

    def _open_output_dir(self):
        """Opens destination directory in Windows Explorer."""
        curr = self.out_dir_var.get()
        if os.path.isdir(curr):
            os.startfile(curr)
        else:
            messagebox.showinfo("Folder Not Found", f"Directory does not exist yet:\n{curr}")

    def _clear_form(self):
        """Resets inputs to defaults."""
        if self.is_running:
            return
        self.url_var.set("")
        self.playlist_start_var.set("")
        self.playlist_end_var.set("")
        self.progress_var.set(0.0)
        self.status_text_var.set("Form reset.")

    def _clear_log(self):
        """Clears the live terminal log."""
        self.txt_log.configure(state="normal")
        self.txt_log.delete("1.0", "end")
        self.txt_log.configure(state="disabled")

    def _copy_log(self):
        """Copies terminal log to clipboard."""
        try:
            content = self.txt_log.get("1.0", "end").strip()
            self.root.clipboard_clear()
            self.root.clipboard_append(content)
            messagebox.showinfo("Copied", "Log contents copied to clipboard.")
        except Exception:
            pass

    def _on_windnd_drop(self, files):
        """Handles OS file drag and drop (e.g. .txt file containing links)."""
        if not files:
            return
        first = files[0]
        if isinstance(first, bytes):
            first = first.decode('utf-8', errors='ignore')
        if os.path.isfile(first) and first.endswith(('.txt', '.csv')):
            try:
                with open(first, 'r', encoding='utf-8') as f:
                    urls = []
                    for line in f:
                        line = line.strip()
                        if line.startswith(('http://', 'https://')):
                            urls.append(line)
                    if urls:
                        self.url_var.set(", ".join(urls))
                        self._append_log(f"[DRAG-DROP] Imported {len(urls)} URL(s) from dropped file.")
                        self._fetch_metadata_inspect()
            except Exception:
                pass

    def _bind_shortcuts(self):
        """Binds universal keyboard accelerators."""
        self.root.bind("<F5>", lambda e: self._start_execution())
        self.root.bind("<Control-Return>", lambda e: self._start_execution())
        self.root.bind("<Escape>", lambda e: self._cancel_execution())
        self.root.bind("<Control-o>", lambda e: self._browse_output_dir())
        self.root.bind("<F1>", lambda e: self._show_shortcuts_modal())

        # Mode switching Alt+1 to Alt+5
        for idx, (m_key, _, _) in enumerate(self.MODES, 1):
            self.root.bind(f"<Alt-Key-{idx}>", lambda e, k=m_key: self._switch_mode(k))

    def _show_shortcuts_modal(self):
        """Displays formatted keyboard shortcuts dialog."""
        modal = tk.Toplevel(self.root)
        modal.title("Keyboard Shortcuts Reference")
        modal.geometry("540x440")
        modal.resizable(False, False)
        modal.configure(bg=self.COLOR_BG)
        modal.transient(self.root)
        modal.grab_set()

        lbl_hdr = tk.Label(
            modal,
            text="Universal Keyboard Shortcuts",
            font=("Segoe UI", 12, "bold"),
            fg=self.COLOR_TEXT,
            bg=self.COLOR_BG
        )
        lbl_hdr.pack(pady=(16, 8))

        content_frame = tk.Frame(modal, bg=self.COLOR_CARD, highlightthickness=1, highlightbackground=self.COLOR_CARD_BORDER)
        content_frame.pack(fill="both", expand=True, padx=16, pady=8)

        shortcuts = [
            ("F5 or Ctrl+Enter", "Load playlist preview or execute download"),
            ("Escape", "Cancel / Abort running operation"),
            ("Ctrl+O", "Browse target output folder"),
            ("Alt+1", "Switch to Playlist Downloader"),
            ("Alt+2", "Switch to Single Video Downloader"),
            ("Alt+3", "Switch to Batch Audio Extractor"),
            ("Alt+4", "Switch to Metadata Inspector & Picker"),
            ("Alt+5", "Switch to Direct CDN Link Exporter"),
            ("F1", "Open this Keyboard Shortcuts Reference"),
            ("Double-Click / Space", "Toggle item selection [YES]/[NO] in table")
        ]

        for k, d in shortcuts:
            row = tk.Frame(content_frame, bg=self.COLOR_CARD)
            row.pack(fill="x", padx=12, pady=4)
            lbl_k = tk.Label(row, text=k, font=("Segoe UI", 9, "bold"), fg="#38BDF8", bg=self.COLOR_CARD, width=20, anchor="w")
            lbl_k.pack(side="left")
            lbl_d = tk.Label(row, text=d, font=("Segoe UI", 9), fg=self.COLOR_TEXT, bg=self.COLOR_CARD, anchor="w")
            lbl_d.pack(side="left")

        btn_close = tk.Button(
            modal,
            text="Close (Esc)",
            font=("Segoe UI", 9, "bold"),
            fg=self.COLOR_TEXT,
            bg=self.COLOR_PRIMARY,
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_PRIMARY_HOVER,
            cursor="hand2",
            bd=0,
            padx=16,
            pady=6,
            command=modal.destroy
        )
        btn_close.pack(pady=12)
        modal.bind("<Escape>", lambda e: modal.destroy())


# ==============================================================================
# ENTRY POINT
# ==============================================================================


    # --------------------------------------------------------------------------
    # INDIVIDUAL VIDEO PROGRESS MONITOR MODAL
    # --------------------------------------------------------------------------

    def _show_tree_context_menu(self, event):
        """Displays right-click context menu on treeview row."""
        iid = self.tree.identify_row(event.y)
        if iid:
            self.tree.selection_set(iid)
            try:
                idx = int(iid) - 1
                if 0 <= idx < len(self.inspected_entries):
                    self._select_inspector_video(idx)
            except ValueError:
                pass
            self.tree_context_menu.tk_popup(event.x_root, event.y_root)

    def _open_individual_progress_modal(self):
        """Opens real-time Individual Video Progress Monitor modal dialog."""
        if self.progress_modal and self.progress_modal.winfo_exists():
            self.progress_modal.lift()
            self.progress_modal.focus_force()
            return

        self.progress_modal = tk.Toplevel(self.root)
        self.progress_modal.title("Real-Time Download Task Monitor -- Individual Video Progress")
        self.progress_modal.geometry("820x540")
        self.progress_modal.minsize(680, 420)
        self.progress_modal.configure(bg=self.COLOR_BG)

        # Header Frame
        hdr = tk.Frame(self.progress_modal, bg=self.COLOR_CARD, highlightthickness=1, highlightbackground=self.COLOR_CARD_BORDER)
        hdr.pack(fill="x", padx=12, pady=(10, 6))

        lbl_hdr_title = tk.Label(
            hdr,
            text="REAL-TIME DOWNLOAD TASK MONITOR",
            font=("Segoe UI", 11, "bold"),
            fg=self.COLOR_TEXT,
            bg=self.COLOR_CARD
        )
        lbl_hdr_title.pack(anchor="w", padx=12, pady=(8, 2))

        self.lbl_modal_summary = tk.Label(
            hdr,
            text="Live metrics for individual concurrent downloading video streams.",
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT_SECONDARY,
            bg=self.COLOR_CARD
        )
        self.lbl_modal_summary.pack(anchor="w", padx=12, pady=(0, 6))

        # Overall progress bar in modal
        self.modal_overall_pbar = ttk.Progressbar(
            hdr,
            variable=self.progress_var,
            maximum=100.0,
            mode="determinate"
        )
        self.modal_overall_pbar.pack(fill="x", padx=12, pady=(0, 8))

        # Scrollable Cards Container
        container = tk.Frame(self.progress_modal, bg=self.COLOR_BG)
        container.pack(fill="both", expand=True, padx=12, pady=4)

        canvas = tk.Canvas(container, bg=self.COLOR_BG, highlightthickness=0)
        scrollbar = ttk.Scrollbar(container, orient="vertical", command=canvas.yview)
        self.modal_cards_frame = tk.Frame(canvas, bg=self.COLOR_BG)

        self.modal_cards_frame.bind(
            "<Configure>",
            lambda e: canvas.configure(scrollregion=canvas.bbox("all"))
        )
        card_window = canvas.create_window((0, 0), window=self.modal_cards_frame, anchor="nw")

        def _on_canvas_configure(event):
            canvas.itemconfig(card_window, width=event.width)
        canvas.bind("<Configure>", _on_canvas_configure)
        canvas.configure(yscrollcommand=scrollbar.set)

        canvas.pack(side="left", fill="both", expand=True)
        scrollbar.pack(side="right", fill="y")

        # Mousewheel scrolling
        def _on_mousewheel(event):
            canvas.yview_scroll(int(-1 * (event.delta / 120)), "units")
        canvas.bind_all("<MouseWheel>", _on_mousewheel)

        self.progress_modal_items = {}
        self._build_progress_modal_cards()

        # Bottom Actions Bar
        bottom_bar = tk.Frame(self.progress_modal, bg=self.COLOR_CARD, highlightthickness=1, highlightbackground=self.COLOR_CARD_BORDER)
        bottom_bar.pack(fill="x", padx=12, pady=(4, 10))

        btn_close = tk.Button(
            bottom_bar,
            text="Close Monitor",
            font=("Segoe UI", 9, "bold"),
            fg=self.COLOR_TEXT,
            bg="#334155",
            activeforeground="#FFFFFF",
            activebackground="#475569",
            cursor="hand2",
            bd=0,
            padx=14,
            pady=4,
            command=self.progress_modal.destroy
        )
        btn_close.pack(side="right", padx=10, pady=6)

    def _build_progress_modal_cards(self):
        """Constructs or re-creates individual progress cards for each task."""
        for widget in self.modal_cards_frame.winfo_children():
            widget.destroy()
        self.progress_modal_items.clear()

        # If no active download tasks state, populate from inspected_entries if available
        tasks = self.download_tasks_state
        if not tasks and self.inspected_entries:
            tasks = {
                e["idx"]: {
                    "idx": e["idx"],
                    "title": e["title"],
                    "quality": e.get("quality", "1080p"),
                    "status": "ready",
                    "percent": 0.0,
                    "downloaded_bytes": 0,
                    "total_bytes": e.get("bytes", 0),
                    "speed": 0.0,
                    "eta": 0
                }
                for e in self.inspected_entries if e.get("selected")
            }

        if not tasks:
            lbl_empty = tk.Label(
                self.modal_cards_frame,
                text="No download tasks currently in queue. Select videos and start batch download.",
                font=("Segoe UI", 9),
                fg=self.COLOR_TEXT_MUTED,
                bg=self.COLOR_BG,
                pady=30
            )
            lbl_empty.pack()
            return

        for idx, task in sorted(tasks.items()):
            card = tk.Frame(self.modal_cards_frame, bg=self.COLOR_CARD, highlightthickness=1, highlightbackground=self.COLOR_CARD_BORDER)
            card.pack(fill="x", pady=3, padx=2)

            # Row 1: Title, Quality badge, Status badge
            row1 = tk.Frame(card, bg=self.COLOR_CARD)
            row1.pack(fill="x", padx=10, pady=(6, 2))

            lbl_title = tk.Label(
                row1,
                text=f"#{idx:02d}  {task.get('title', 'Unknown')[:60]}",
                font=("Segoe UI", 9, "bold"),
                fg=self.COLOR_TEXT,
                bg=self.COLOR_CARD,
                anchor="w"
            )
            lbl_title.pack(side="left", fill="x", expand=True)

            lbl_qual = tk.Label(
                row1,
                text=str(task.get("quality", "")),
                font=("Segoe UI", 8, "bold"),
                fg="#38BDF8",
                bg="#0F172A",
                padx=6,
                pady=1,
                relief="solid",
                bd=1
            )
            lbl_qual.pack(side="left", padx=6)

            status_str = task.get("status", "QUEUED").upper()
            status_bg = "#10B981" if status_str == "COMPLETED" else ("#0284C7" if status_str == "DOWNLOADING" else "#334155")
            lbl_status = tk.Label(
                row1,
                text=status_str,
                font=("Segoe UI", 8, "bold"),
                fg="#FFFFFF",
                bg=status_bg,
                padx=8,
                pady=1
            )
            lbl_status.pack(side="right")

            # Row 2: Progress bar
            pbar = ttk.Progressbar(card, maximum=100.0, mode="determinate")
            pbar.pack(fill="x", padx=10, pady=3)
            pbar["value"] = task.get("percent", 0.0)

            # Row 3: Readouts
            row3 = tk.Frame(card, bg=self.COLOR_CARD)
            row3.pack(fill="x", padx=10, pady=(0, 6))

            pct = task.get("percent", 0.0)
            dl_b = task.get("downloaded_bytes", 0)
            tot_b = task.get("total_bytes", 0)
            speed = task.get("speed", 0.0)
            eta = task.get("eta", 0)

            readout_str = f"{pct:.1f}% | Size: {format_bytes(dl_b)} / {format_bytes(tot_b)} | Speed: {format_bytes(speed)}/s | ETA: {format_seconds(eta)}"
            lbl_readout = tk.Label(
                row3,
                text=readout_str,
                font=("Consolas", 8),
                fg=self.COLOR_TEXT_SECONDARY,
                bg=self.COLOR_CARD
            )
            lbl_readout.pack(side="left")

            self.progress_modal_items[idx] = {
                "pbar": pbar,
                "lbl_status": lbl_status,
                "lbl_readout": lbl_readout
            }

    def _update_progress_modal_if_open(self):
        """Refreshes individual card values in the open Progress Monitor modal."""
        if not self.progress_modal or not self.progress_modal.winfo_exists():
            return

        if not self.progress_modal_items and self.download_tasks_state:
            self._build_progress_modal_cards()
            return

        total_tasks = len(self.download_tasks_state)
        active_c = sum(1 for t in self.download_tasks_state.values() if t.get("status") == "downloading")
        done_c = sum(1 for t in self.download_tasks_state.values() if t.get("status") == "completed")

        if hasattr(self, "lbl_modal_summary"):
            self.lbl_modal_summary.configure(
                text=f"Batch Progress: {done_c}/{total_tasks} completed ({active_c} active) | {self.speed_text_var.get()}"
            )

        for idx, task in self.download_tasks_state.items():
            if idx not in self.progress_modal_items:
                continue
            widgets = self.progress_modal_items[idx]
            pct = task.get("percent", 0.0)
            widgets["pbar"]["value"] = pct

            status_str = task.get("status", "QUEUED").upper()
            status_bg = "#10B981" if status_str == "COMPLETED" else ("#0284C7" if status_str == "DOWNLOADING" else "#334155")
            widgets["lbl_status"].configure(text=status_str, bg=status_bg)

            dl_b = task.get("downloaded_bytes", 0)
            tot_b = task.get("total_bytes", 0)
            speed = task.get("speed", 0.0)
            eta = task.get("eta", 0)
            readout_str = f"{pct:.1f}% | Size: {format_bytes(dl_b)} / {format_bytes(tot_b)} | Speed: {format_bytes(speed)}/s | ETA: {format_seconds(eta)}"
            widgets["lbl_readout"].configure(text=readout_str)

    # --------------------------------------------------------------------------
    # CUSTOM PROFESSIONAL VIDEO & AUDIO MEDIA PLAYER MODAL
    # --------------------------------------------------------------------------

    def _open_custom_media_player(self, entry=None):
        """Opens the custom enterprise video/audio preview player modal."""
        # Resolve target entry
        if entry is None:
            sel = self.tree.selection()
            if sel:
                try:
                    idx = int(sel[0]) - 1
                    if 0 <= idx < len(self.inspected_entries):
                        entry = self.inspected_entries[idx]
                except ValueError:
                    pass

        if entry is None and self.active_entry_idx is not None and self.active_entry_idx < len(self.inspected_entries):
            entry = self.inspected_entries[self.active_entry_idx]

        if entry is None:
            raw_url = self.url_var.get().strip()
            if raw_url:
                entry = {
                    "title": "Direct Stream URL Preview",
                    "uploader": "YouTube Stream",
                    "duration": "Live / On-Demand",
                    "duration_sec": 180,
                    "quality": self.res_var.get(),
                    "url": raw_url,
                    "idx": 1
                }

        if not entry:
            messagebox.showinfo("Select Media", "Please select a video or audio row from the table to preview.")
            return

        if self.media_player_modal and self.media_player_modal.winfo_exists():
            self.media_player_modal.destroy()

        self.media_player_modal = tk.Toplevel(self.root)
        title = entry.get("title", "Media Preview")
        self.media_player_modal.title(f"Media Player Pro -- {title[:45]}")
        self.media_player_modal.geometry("740x580")
        self.media_player_modal.minsize(640, 480)
        self.media_player_modal.configure(bg=self.COLOR_BG)

        # Initialize WMP COM object
        wmp = None
        try:
            import win32com.client
            wmp = win32com.client.Dispatch("WMPlayer.OCX")
            wmp.settings.autoStart = False
            wmp.settings.volume = 80
        except Exception as e:
            self._append_log(f"[PLAYER] WMP engine notice: {e}")

        # Top Banner
        hdr = tk.Frame(self.media_player_modal, bg=self.COLOR_CARD, highlightthickness=1, highlightbackground=self.COLOR_CARD_BORDER)
        hdr.pack(fill="x", padx=12, pady=(10, 4))

        lbl_t = tk.Label(
            hdr,
            text=f"{title[:65]}",
            font=("Segoe UI", 11, "bold"),
            fg=self.COLOR_TEXT,
            bg=self.COLOR_CARD
        )
        lbl_t.pack(anchor="w", padx=12, pady=(8, 1))

        sub_info = f"Channel: {entry.get('uploader', 'Unknown')}  |  Duration: {entry.get('duration', 'N/A')}  |  Preset: {entry.get('quality', '1080p')}"
        lbl_sub = tk.Label(
            hdr,
            text=sub_info,
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT_SECONDARY,
            bg=self.COLOR_CARD
        )
        lbl_sub.pack(anchor="w", padx=12, pady=(0, 8))

        # Media Visualizer & Artwork Stage Canvas
        stage_frame = tk.Frame(self.media_player_modal, bg=self.COLOR_CARD, highlightthickness=1, highlightbackground=self.COLOR_CARD_BORDER)
        stage_frame.pack(fill="both", expand=True, padx=12, pady=4)

        canvas = tk.Canvas(stage_frame, bg="#050914", highlightthickness=0)
        canvas.pack(fill="both", expand=True, padx=4, pady=4)

        # Load thumbnail artwork onto canvas if available
        thumb_img = None
        idx = entry.get("idx", 1)
        if idx in self.thumbnail_cache and PILLOW_AVAILABLE:
            thumb_img = self.thumbnail_cache[idx]

        # Equalizer simulation bars
        eq_bars = [10] * 32
        anim_running = [True]
        is_playing = [False]
        resolved_url = [None]

        def draw_stage():
            if not canvas.winfo_exists():
                return
            w = canvas.winfo_width()
            h = canvas.winfo_height()
            if w <= 1 or h <= 1:
                self.root.after(50, draw_stage)
                return

            canvas.delete("all")

            # Draw background grid
            for gy in range(0, h, 30):
                canvas.create_line(0, gy, w, gy, fill="#0F172A", width=1)

            # Center preview placeholder or artwork
            center_x = w // 2
            center_y = h // 2

            if is_playing[0]:
                # Draw dynamic animated cyber audio spectrum visualizer
                import random
                num_bars = len(eq_bars)
                bar_width = max(4, (w - 120) // num_bars)
                start_x = (w - (num_bars * (bar_width + 3))) // 2

                for i in range(num_bars):
                    target_h = random.randint(15, int(h * 0.75))
                    eq_bars[i] = int(eq_bars[i] * 0.6 + target_h * 0.4)
                    bx = start_x + i * (bar_width + 3)
                    by2 = h - 35
                    by1 = by2 - eq_bars[i]
                    # Gradient color simulation
                    fill_col = "#38BDF8" if i % 2 == 0 else "#818CF8"
                    canvas.create_rectangle(bx, by1, bx + bar_width, by2, fill=fill_col, outline="")

                canvas.create_text(center_x, 40, text="PLAYING MEDIA STREAM", fill="#38BDF8", font=("Segoe UI", 11, "bold"))
            else:
                canvas.create_text(center_x, center_y - 20, text="MEDIA READY FOR PREVIEW", fill="#64748B", font=("Segoe UI", 12, "bold"))
                canvas.create_text(center_x, center_y + 15, text="Click [ PLAY ] below to start streaming playback", fill="#475569", font=("Segoe UI", 9))

            # Bottom time display
            cur_pos = 0
            tot_dur = entry.get("duration_sec", 180)
            if wmp:
                try:
                    cur_pos = wmp.controls.currentPosition
                    if wmp.currentMedia and wmp.currentMedia.duration > 0:
                        tot_dur = wmp.currentMedia.duration
                except Exception:
                    pass
            canvas.create_text(
                center_x,
                h - 15,
                text=f"{format_seconds(cur_pos)} / {format_seconds(tot_dur)}",
                fill="#94A3B8",
                font=("Consolas", 9, "bold")
            )

            if anim_running[0]:
                self.root.after(100, draw_stage)

        self.root.after(50, draw_stage)

        # Control Deck Frame
        deck = tk.Frame(self.media_player_modal, bg=self.COLOR_CARD, highlightthickness=1, highlightbackground=self.COLOR_CARD_BORDER)
        deck.pack(fill="x", padx=12, pady=(4, 6))

        # Row 1: Seek Slider
        seek_row = tk.Frame(deck, bg=self.COLOR_CARD)
        seek_row.pack(fill="x", padx=12, pady=(8, 4))

        seek_var = tk.DoubleVar(value=0.0)
        tot_dur_sec = max(1, entry.get("duration_sec", 180))
        seek_scale = ttk.Scale(seek_row, variable=seek_var, from_=0.0, to=float(tot_dur_sec), orient="horizontal", cursor="hand2")
        seek_scale.pack(fill="x")

        def _on_seek_released(event):
            if wmp:
                try:
                    wmp.controls.currentPosition = seek_var.get()
                except Exception:
                    pass
        seek_scale.bind("<ButtonRelease-1>", _on_seek_released)

        # Row 2: Transport Buttons, Volume, Options
        ctrl_row = tk.Frame(deck, bg=self.COLOR_CARD)
        ctrl_row.pack(fill="x", padx=12, pady=(4, 8))

        btn_play = tk.Button(
            ctrl_row,
            text="PLAY",
            font=("Segoe UI", 9, "bold"),
            fg="#FFFFFF",
            bg=self.COLOR_SUCCESS,
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_SUCCESS_HOVER,
            cursor="hand2",
            bd=0,
            padx=16,
            pady=4
        )
        btn_play.pack(side="left", padx=(0, 6))

        btn_stop = tk.Button(
            ctrl_row,
            text="STOP",
            font=("Segoe UI", 9, "bold"),
            fg=self.COLOR_TEXT,
            bg="#334155",
            activeforeground="#FFFFFF",
            activebackground="#475569",
            cursor="hand2",
            bd=0,
            padx=12,
            pady=4
        )
        btn_stop.pack(side="left", padx=4)

        btn_rew = tk.Button(
            ctrl_row,
            text="-10s",
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT,
            bg="#1E293B",
            cursor="hand2",
            bd=0,
            padx=8,
            pady=4,
            command=lambda: _seek_offset(-10)
        )
        btn_rew.pack(side="left", padx=2)

        btn_fwd = tk.Button(
            ctrl_row,
            text="+10s",
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT,
            bg="#1E293B",
            cursor="hand2",
            bd=0,
            padx=8,
            pady=4,
            command=lambda: _seek_offset(10)
        )
        btn_fwd.pack(side="left", padx=2)

        def _seek_offset(delta):
            if wmp:
                try:
                    new_pos = max(0, wmp.controls.currentPosition + delta)
                    wmp.controls.currentPosition = new_pos
                    seek_var.set(new_pos)
                except Exception:
                    pass

        # Volume Slider
        lbl_vol = tk.Label(ctrl_row, text="Vol:", font=("Segoe UI", 8, "bold"), fg=self.COLOR_TEXT_SECONDARY, bg=self.COLOR_CARD)
        lbl_vol.pack(side="left", padx=(14, 2))

        vol_var = tk.IntVar(value=80)
        vol_scale = ttk.Scale(ctrl_row, variable=vol_var, from_=0, to=100, orient="horizontal", length=80, cursor="hand2")
        vol_scale.pack(side="left", padx=2)

        def _on_vol_changed(*args):
            if wmp:
                try:
                    wmp.settings.volume = int(vol_var.get())
                except Exception:
                    pass
        vol_var.trace_add("write", _on_vol_changed)

        # External Player & YouTube buttons
        btn_ext = tk.Button(
            ctrl_row,
            text="Open External Player",
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT,
            bg="#1E293B",
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_PRIMARY,
            cursor="hand2",
            bd=0,
            padx=10,
            pady=4,
            command=lambda: _launch_external()
        )
        btn_ext.pack(side="right", padx=4)

        btn_yt = tk.Button(
            ctrl_row,
            text="Open on YouTube",
            font=("Segoe UI", 8),
            fg=self.COLOR_TEXT,
            bg="#1E293B",
            activeforeground="#FFFFFF",
            activebackground=self.COLOR_PRIMARY,
            cursor="hand2",
            bd=0,
            padx=10,
            pady=4,
            command=lambda: webbrowser.open(entry.get("url", ""))
        )
        btn_yt.pack(side="right", padx=4)

        def _launch_external():
            raw_target = resolved_url[0] or entry.get("url", "")
            if raw_target:
                try:
                    os.startfile(raw_target)
                except Exception:
                    webbrowser.open(raw_target)

        def _toggle_play():
            if not is_playing[0]:
                _start_playback()
            else:
                _pause_playback()

        def _start_playback():
            target_url = entry.get("url")
            # Check if local downloaded file exists
            out_dir = self.out_dir_var.get().strip()
            local_cand = None
            if os.path.isdir(out_dir):
                for f in os.listdir(out_dir):
                    if entry.get("title", "")[:20].lower() in f.lower():
                        local_cand = os.path.join(out_dir, f)
                        break

            if local_cand and os.path.exists(local_cand):
                resolved_url[0] = local_cand
                self._append_log(f"[PLAYER] Playing local downloaded file: {local_cand}")
                if wmp:
                    try:
                        wmp.URL = local_cand
                        wmp.controls.play()
                        is_playing[0] = True
                        btn_play.configure(text="PAUSE", bg="#D97706")
                    except Exception as e:
                        self._append_log(f"[PLAYER] Play error: {e}")
            else:
                # Fetch stream URL via background worker if not yet resolved
                if not resolved_url[0]:
                    self._append_log(f"[PLAYER] Resolving stream URL for preview: {target_url}...")

                    def resolve_worker():
                        try:
                            if YTDLP_AVAILABLE and yt_dlp:
                                with yt_dlp.YoutubeDL({'quiet': True, 'no_warnings': True}) as ydl:
                                    info = ydl.extract_info(target_url, download=False)
                                    formats = info.get('formats') or []
                                    cands = [f for f in formats if f.get('url')]
                                    best = cands[0] if cands else None
                                    stream_url = best.get('url') if best else info.get('url')
                                    resolved_url[0] = stream_url

                                if resolved_url[0] and wmp:
                                    wmp.URL = resolved_url[0]
                                    wmp.controls.play()
                                    is_playing[0] = True
                                    self.root.after(0, lambda: btn_play.configure(text="PAUSE", bg="#D97706"))
                        except Exception as err:
                            self.ui_queue.put(("log", f"[PLAYER] Stream extraction failed: {err}"))
                            self.root.after(0, lambda: webbrowser.open(target_url))

                    threading.Thread(target=resolve_worker, daemon=True).start()
                elif wmp:
                    wmp.controls.play()
                    is_playing[0] = True
                    btn_play.configure(text="PAUSE", bg="#D97706")

        def _pause_playback():
            if wmp:
                try:
                    wmp.controls.pause()
                except Exception:
                    pass
            is_playing[0] = False
            btn_play.configure(text="PLAY", bg=self.COLOR_SUCCESS)

        def _stop_playback():
            if wmp:
                try:
                    wmp.controls.stop()
                    wmp.controls.currentPosition = 0
                except Exception:
                    pass
            is_playing[0] = False
            seek_var.set(0.0)
            btn_play.configure(text="PLAY", bg=self.COLOR_SUCCESS)

        btn_play.configure(command=_toggle_play)
        btn_stop.configure(command=_stop_playback)

        # Update position loop
        def update_timer():
            if not self.media_player_modal or not self.media_player_modal.winfo_exists():
                return
            if wmp and is_playing[0]:
                try:
                    pos = wmp.controls.currentPosition
                    seek_var.set(pos)
                except Exception:
                    pass
            self.root.after(500, update_timer)

        self.root.after(500, update_timer)

        def _on_close():
            anim_running[0] = False
            _stop_playback()
            if wmp:
                try:
                    wmp.close()
                except Exception:
                    pass
            self.media_player_modal.destroy()

        self.media_player_modal.protocol("WM_DELETE_WINDOW", _on_close)


def launch_gui():
    """Launches the YouTube Media Suite & Playlist Toolkit Pro Desktop GUI."""
    app = YouTubeDownloaderGUI()
    app.root.mainloop()


if __name__ == "__main__":
    launch_gui()
